In [ ]:
# Cell 1: 라이브러리 설치
!pip install pandas tqdm openai
!pip install ipywidgets
!pip install mlxtend


In [1]:
!pip install mlxtend

In [ ]:
# Cell 2 (수정본): CSV 병합 + 정제
import os
import glob
import pandas as pd

DATA_DIR   = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\1661 - 1722"
OUTPUT_DIR = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_CSV = os.path.join(OUTPUT_DIR, "merged.csv")

# ───── 사전 제거할 명백한 허사 (1글자) ─────
# NER API 비용 절감용. 어차피 NONE으로 분류될 게 확실한 한자.
STOP_CHARS = set([
    "之", "也", "以", "而", "於", "其", "者", "焉", "乎", "矣",
    "曰", "云", "等", "及", "與", "或", "亦", "又", "則", "且",
    "於", "為", "為", "以", "於", "之", "於",
    "是", "此", "彼", "斯", "茲",
    "上", "下", "中", "內", "外",
    "有", "無", "不", "未", "非", "勿", "毋",
    "可", "能", "得", "當", "宜", "應",
    "皆", "俱", "悉", "盡", "並", "共",
    "甚", "至", "極", "最",
    "已", "既", "嘗", "曾",
    "一", "二", "三", "四", "五", "六", "七", "八", "九", "十",
    "百", "千", "萬",
])

all_dfs = []
csv_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))
print(f"[INFO] 발견된 파일 수: {len(csv_files)}개")

skipped_files = 0
for filepath in csv_files:
    filename = os.path.basename(filepath)
    name     = filename.replace(".csv", "")

    try:
        parts = name.split("_")
        year = int(parts[0])

        if parts[1].startswith("yun"):
            is_yun = True
            month  = int(parts[1].replace("yun", ""))
        else:
            is_yun = False
            month  = int(parts[1])
    except (ValueError, IndexError):
        print(f"[WARN] 파일명 형식 불일치, 건너뜀: {filename}")
        skipped_files += 1
        continue

    try:
        df = pd.read_csv(filepath, encoding="utf-8-sig")
    except UnicodeDecodeError:
        df = pd.read_csv(filepath, encoding="cp949")

    df.columns = ["word", "freq"]

    # ───── 정제 ─────
    # 1. word NaN 제거
    df = df.dropna(subset=["word"])
    # 2. word를 문자열로 강제 + 공백 제거
    df["word"] = df["word"].astype(str).str.strip()
    # 3. 빈 문자열 제거
    df = df[df["word"] != ""]
    # 4. freq 숫자 변환 + 양수만
    df["freq"] = pd.to_numeric(df["freq"], errors="coerce")
    df = df.dropna(subset=["freq"])
    df = df[df["freq"] > 0]
    # 5. freq를 정수로
    df["freq"] = df["freq"].astype(int)
    # 6. 1글자 허사 제거
    df = df[~df["word"].isin(STOP_CHARS)]

    # 메타 컬럼
    df["year"]        = year
    df["month"]       = month
    df["is_yun"]      = is_yun
    df["period_year"] = str(year)
    if is_yun:
        df["period_month"] = f"{year:04d}-yun{month:02d}"
    else:
        df["period_month"] = f"{year:04d}-{month:02d}"

    all_dfs.append(df)

merged = pd.concat(all_dfs, ignore_index=True)

# ───── 같은 (period_month, word) 중복 시 freq 합산 ─────
before = len(merged)
merged = (
    merged.groupby(
        ["year", "month", "is_yun", "period_year", "period_month", "word"],
        as_index=False
    )["freq"].sum()
)
after = len(merged)
if before != after:
    print(f"[INFO] 중복 합산: {before:,} → {after:,}행")

merged = merged[[
    "year", "month", "is_yun",
    "period_year", "period_month",
    "word", "freq"
]].sort_values(["year", "month", "is_yun"]).reset_index(drop=True)

merged.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

# ───── 통계 ─────
print(f"\n[DONE] 병합 완료 → {OUTPUT_CSV}")
print(f"  처리된 파일 : {len(csv_files) - skipped_files:,}개 (건너뜀 {skipped_files}개)")
print(f"  총 행 수    : {len(merged):,}행")
print(f"  기간        : {merged['period_month'].min()} ~ {merged['period_month'].max()}")
print(f"  고유 단어   : {merged['word'].nunique():,}개")
print(f"  일반 달     : {merged[~merged['is_yun']]['period_month'].nunique():,}개월")
print(f"  윤달        : {merged[merged['is_yun']]['period_month'].nunique():,}개월")

# 단어 길이 분포
merged["wlen"] = merged["word"].str.len()
print(f"\n[단어 길이 분포 (고유 단어 기준)]")
unique_words = merged.drop_duplicates("word")
print(unique_words["wlen"].value_counts().sort_index().to_string())

print(f"\n[샘플]")
print(merged.drop(columns=["wlen"]).head(10).to_string(index=False))


In [ ]:
# Cell 3 (수정본): NER 설정
import os
import re
import json
import time
import pandas as pd
from tqdm import tqdm
from openai import OpenAI
from dotenv import load_dotenv 

INPUT_CSV      = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\merged.csv"
OUTPUT_ALL_CSV = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\ner_results_all.csv"
OUTPUT_CSV     = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\ner_results.csv"
CHECKPOINT_CSV = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\ner_checkpoint.csv"
load_dotenv(r"C:\Users\USER\OneDrive\바탕 화면\nerproject\.env")

# ───── API 설정 ─────
DEEPSEEK_API_KEY  = os.environ.get("DEEPSEEK_API_KEY", "")
DEEPSEEK_BASE_URL = "https://api.deepseek.com"
MODEL_NAME        = "deepseek-v4-flash"  # ← deepseek-v4-pro로 변경도 고려 (역사적 맥락을 이해하지 못하는 경우, 다하고 지워라 빨리)

# ───── 실행 파라미터 ─────
BATCH_SIZE   = 30 # 혹은 50
DELAY_API    = 1.5
CHECKPOINT_N = 10

# ───── 검증 ─────
if DEEPSEEK_API_KEY:
    masked = DEEPSEEK_API_KEY[:7] + "..." + DEEPSEEK_API_KEY[-4:]
    print(f"[OK] API 키 로드 완료: {masked}")
else:
    print("[ERROR] API 키를 .env에서 못 찾음. .env 파일 위치와 변수명 확인하세요.")

client = OpenAI(
    api_key=DEEPSEEK_API_KEY or "PLACEHOLDER",
    base_url=DEEPSEEK_BASE_URL
)

SYSTEM_PROMPT = """너는 청나라 실록(淸實錄) 한문 단어를 분류하는 전문가다.
주어진 한문 단어 목록 각각을 아래 카테고리 중 하나로 분류하라.

【카테고리】
- PER  : 인물 (고유 인명. 황제 본명, 신하, 장군 등)
- LOC  : 지명 (지역, 도시, 궁궐, 강, 산 등)
- OFF  : 관직/직책/작위 (大學士, 總督, 巡撫, 貝勒, 公 등)
- GRP  : 민족/집단/세력/부족 (蒙古, 準噶爾, 三藩, 科爾沁 등)
- DATE : 연호/날짜/간지 (康熙, 正月, 辛亥, 甲子 등)
- SYS  : 제도/법령/정책 (科擧, 海禁, 八旗 등)
- EVT  : 사건/전쟁/반란 (三藩之亂 등)
- NONE : 위 카테고리에 해당 없음 (일반어, 동사, 조사, 모호한 표현 등)

【분류 예시 — 정상 케이스】
- "索尼"        → PER
- "鰲拜"        → PER
- "遏必隆"      → PER
- "麻勒吉"      → PER
- "大學士索尼"  → PER  (관직+인명 결합형은 PER)
- "總督于成龍"  → PER  (관직+인명 결합형은 PER)
- "大學士"      → OFF  (관직 단독)
- "巡撫"        → OFF
- "侍郎"        → OFF
- "貝勒"        → OFF  (작위)
- "公"          → OFF  (작위)
- "盛京"        → LOC
- "乾清門"      → LOC  (궁궐)
- "黃河"        → LOC
- "蒙古"        → GRP
- "準噶爾"      → GRP
- "科爾沁"      → GRP  (몽골 부족)
- "三藩"        → GRP
- "康熙"        → DATE
- "正月"        → DATE
- "辛亥"        → DATE  (간지)
- "科擧"        → SYS
- "海禁"        → SYS
- "三藩之亂"    → EVT

【분류 예시 — 주의해야 할 오류 케이스】
- "大行"        → NONE  ('大行皇帝(승하한 황제)' 일부, 고유명사 아님)
- "今上"        → NONE  ('현 황제' 일반 지칭, 고유명사 아님)
- "皇上"        → NONE  (일반 호칭)
- "上"          → NONE  (단독으로는 모호)
- "帝"          → NONE  (직위 일반어)
- "世祖"        → DATE  (묘호. 특정 황제 지칭이지만 고유명사성 약함 — 묘호는 DATE로)
- "王貝勒"      → OFF   (작위 나열, 인명 아님)
- "王公"        → OFF   (작위 나열)
- "四子"        → NONE  ('넷째 아들' 일반어. '四子部落' 같은 부족명 아니면 NONE)
- "宗室"        → GRP   (황실 종친 집단)
- "滿漢"        → GRP   (만주족+한족)
- "蘇克薩哈遏"  → NONE  (두 인명 '蘇克薩哈'+'遏必隆'이 잘못 붙음 → NONE)
- "任學士"      → NONE  ('任'이 동사 '임명하다'일 가능성 → NONE)
- "自慎以"      → NONE  (토크나이저 오류 조각)
- "之後萬"      → NONE  (토크나이저 오류 조각)
- "海面"        → NONE  (일반명사)
- "流民"        → NONE  (일반명사)
- "百姓"        → NONE  (일반명사)
- "朝廷"        → NONE  (일반명사)

【분류 규칙】
1. 반드시 JSON만 반환. 설명 텍스트 절대 없음.
2. 형식: {"단어": "카테고리", "단어": "카테고리", ...}
3. 입력된 단어를 빠짐없이 전부 분류.
4. 카테고리 값은 반드시 PER/LOC/OFF/GRP/DATE/SYS/EVT/NONE 중 하나.
5. 관직+인명 결합형(예: '大學士某某')은 PER.
6. 두 인명이 잘못 붙은 것 같은 단어(어색한 음역 결합)는 NONE.
7. 황제 일반 지칭(大行/今上/皇上/帝/上)은 NONE.
8. 묘호(世祖/聖祖/太祖)는 DATE.
9. 일반명사(百姓/朝廷/流民/海面)는 NONE.
10. 토크나이저 오류로 보이는 의미 불명 조각은 NONE.
11. 확신이 없으면 NONE.
"""

print("[INFO] 설정 완료")
print(f"  - 모델       : {MODEL_NAME}")
print(f"  - 배치 크기  : {BATCH_SIZE}")
print(f"  - API 대기   : {DELAY_API}초")
print(f"  - 체크포인트 : {CHECKPOINT_N}배치마다")
if not DEEPSEEK_API_KEY:
    print("[WARN] API 키가 비어있습니다. 실행 전에 채워 넣으세요.")


[OK] API 키 로드 완료: sk-2498...b45c
[INFO] 설정 완료
  - 모델       : deepseek-v4-flash
  - 배치 크기  : 30
  - API 대기   : 1.5초
  - 체크포인트 : 10배치마다


In [3]:
# Cell 4 (수정본): NER 함수 정의
VALID_CATEGORIES = {"PER", "LOC", "OFF", "GRP", "DATE", "SYS", "EVT", "NONE"}


def call_ner(words: list, retries: int = 2) -> dict:
    """
    한문 단어 리스트를 카테고리로 분류.
    - 성공: {word: category}
    - API 오류 / JSON 파싱 실패: 해당 단어들을 "UNK"로 반환 (다음 실행 시 재시도)
    - 모델이 일부 단어를 누락하면 누락분만 자동 재요청
    """
    if not DEEPSEEK_API_KEY:
        return {w: "NONE" for w in words}

    word_list = "\n".join(f"- {w}" for w in words)

    for attempt in range(retries + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": f"다음 단어들을 분류하라:\n{word_list}"}
                ],
                temperature=0.0,
                max_tokens=8192, # 혹은 4096
                response_format={"type": "json_object"},  # JSON 강제 (호환 안 되면 이 줄 삭제)
            )
            raw = resp.choices[0].message.content.strip()
            raw = re.sub(r"```json\s*", "", raw)
            raw = re.sub(r"```\s*", "", raw)
            parsed = json.loads(raw)

            # 카테고리 화이트리스트 검증 (PERSON, PLACE 같은 변형 → NONE으로 강제)
            for w, c in list(parsed.items()):
                if c not in VALID_CATEGORIES:
                    parsed[w] = "NONE"

            # 누락 단어 검출 → 누락분만 재요청
            missing = [w for w in words if w not in parsed]
            if missing:
                if attempt < retries:
                    print(f"[RETRY] 누락 {len(missing)}개 재요청")
                    sub = call_ner(missing, retries=0)
                    parsed.update(sub)
                    for w in words:
                        if w not in parsed:
                            parsed[w] = "UNK"
                else:
                    for w in missing:
                        parsed[w] = "UNK"

            return parsed

        except json.JSONDecodeError as e:
            print(f"[WARN] JSON 파싱 실패 (시도 {attempt+1}/{retries+1}): {e}")
            if attempt == retries:
                return {w: "UNK" for w in words}
            time.sleep(2)
        except Exception as e:
            print(f"[WARN] API 오류 (시도 {attempt+1}/{retries+1}): {e}")
            if attempt == retries:
                return {w: "UNK" for w in words}
            time.sleep(3)

    return {w: "UNK" for w in words}


print("[INFO] 함수 정의 완료")


[INFO] 함수 정의 완료


In [4]:
# Cell 5 (수정본): NER 실행 — UNK 자동 재시도 + 안전 저장
df = pd.read_csv(INPUT_CSV)
all_words = df["word"].dropna().astype(str).unique().tolist()
print(f"[INFO] 전체 고유 단어: {len(all_words):,}개")

# 체크포인트 로드
done_map = {}
if os.path.exists(CHECKPOINT_CSV):
    ck = pd.read_csv(CHECKPOINT_CSV)
    done_map = dict(zip(ck["word"].astype(str), ck["category"].astype(str)))
    n_unk = sum(1 for v in done_map.values() if v == "UNK")
    print(f"[INFO] 체크포인트 로드: {len(done_map):,}개 (UNK {n_unk:,}개는 재시도)")

# 처리 대상: 미처리 + UNK
todo_words = [
    w for w in all_words
    if (w not in done_map) or (done_map.get(w) == "UNK")
]
print(f"[INFO] 처리 대상: {len(todo_words):,}개")

batches = [todo_words[i:i+BATCH_SIZE] for i in range(0, len(todo_words), BATCH_SIZE)]
print(f"[INFO] 배치 수: {len(batches)}개\n")


def save_checkpoint(done_map):
    ck_df = pd.DataFrame(list(done_map.items()), columns=["word", "category"])
    ck_df.to_csv(CHECKPOINT_CSV, index=False, encoding="utf-8-sig")
    return ck_df


# ───── 메인 루프 (Ctrl+C 안전) ─────
try:
    for batch_idx, batch in enumerate(tqdm(batches, desc="NER 배치")):
        result = call_ner(batch)
        done_map.update(result)
        time.sleep(DELAY_API)

        if (batch_idx + 1) % CHECKPOINT_N == 0:
            ck_df = save_checkpoint(done_map)
            dist  = ck_df["category"].value_counts().to_dict()
            print(f"[CHECKPOINT] {len(done_map):,}개 저장 / 분포: {dist}")

except KeyboardInterrupt:
    print("\n[INTERRUPT] 사용자 중단 — 진행분 저장 중...")
finally:
    save_checkpoint(done_map)
    print(f"[SAFE] 체크포인트 저장 완료: {len(done_map):,}개")


# ───── 최종 결과 생성 ─────
ner_df = pd.DataFrame(list(done_map.items()), columns=["word", "category"])

n_unk = (ner_df["category"] == "UNK").sum()
if n_unk > 0:
    print(f"\n[WARN] UNK {n_unk:,}개 남음 — Cell 5를 다시 실행하면 자동 재시도합니다.")

# merged에 category 병합
df_out = df.merge(ner_df, on="word", how="left")
df_out["category"] = df_out["category"].fillna("NONE")

# 분석용: NONE / UNK 모두 제외
df_filtered = df_out[~df_out["category"].isin(["NONE", "UNK"])].reset_index(drop=True)

df_out.to_csv(OUTPUT_ALL_CSV, index=False, encoding="utf-8-sig")
df_filtered.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print(f"\n[DONE] NER 완료")
print(f"  전체 (NONE/UNK 포함) → ner_results_all.csv  ({len(df_out):,}행)")
print(f"  분석용 (NONE/UNK 제외) → ner_results.csv      ({len(df_filtered):,}행)")
print(f"\n[카테고리 분포 — 단어 기준 (고유 단어)]")
print(ner_df['category'].value_counts().to_string())
print(f"\n[카테고리 분포 — 행 기준 (분석용 데이터)]")
print(df_filtered['category'].value_counts().to_string())


[INFO] 전체 고유 단어: 35,100개
[INFO] 체크포인트 로드: 26,461개 (UNK 0개는 재시도)
[INFO] 처리 대상: 8,693개
[INFO] 배치 수: 290개



NER 배치:   3%|▎         | 10/290 [03:22<1:43:58, 22.28s/it]

[CHECKPOINT] 26,761개 저장 / 분포: {'NONE': 22922, 'PER': 2012, 'LOC': 891, 'OFF': 436, 'SYS': 175, 'DATE': 173, 'GRP': 146, 'EVT': 6}


NER 배치:   7%|▋         | 20/290 [06:24<1:16:06, 16.91s/it]

[CHECKPOINT] 27,061개 저장 / 분포: {'NONE': 23152, 'PER': 2056, 'LOC': 909, 'OFF': 437, 'DATE': 180, 'SYS': 175, 'GRP': 146, 'EVT': 6}


...

NER 배치:  97%|█████████▋| 280/290 [1:27:32<03:14, 19.49s/it]

[CHECKPOINT] 34,861개 저장 / 분포: {'NONE': 29977, 'PER': 2668, 'LOC': 1123, 'OFF': 484, 'SYS': 221, 'DATE': 196, 'GRP': 183, 'EVT': 9}


NER 배치: 100%|██████████| 290/290 [1:30:10<00:00, 18.66s/it]

[CHECKPOINT] 35,154개 저장 / 분포: {'NONE': 30232, 'PER': 2688, 'LOC': 1131, 'OFF': 486, 'SYS': 221, 'DATE': 200, 'GRP': 186, 'EVT': 10}
[SAFE] 체크포인트 저장 완료: 35,154개



[DONE] NER 완료
  전체 (NONE/UNK 포함) → ner_results_all.csv  (102,861행)
  분석용 (NONE/UNK 제외) → ner_results.csv      (21,126행)

[카테고리 분포 — 단어 기준 (고유 단어)]
category
NONE    30232
PER      2688
LOC      1131
OFF       486
SYS       221
DATE      200
GRP       186
EVT        10

[카테고리 분포 — 행 기준 (분석용 데이터)]
category
OFF     5547
DATE    4761
PER     4406
LOC     4071
SYS     1173
GRP     1139
EVT       29


In [5]:
# Cell 5.1: NER 결과 진단
import pandas as pd

ck = pd.read_csv(CHECKPOINT_CSV)
merged = pd.read_csv(INPUT_CSV)

# ───── 1. 전체 분포 ─────
print("=" * 60)
print("[1] 전체 카테고리 분포")
print("=" * 60)
print(ck['category'].value_counts().to_string())
print(f"\nUNK 잔존: {(ck['category']=='UNK').sum()}개")
print(f"전체 단어: {len(ck):,}개")

# ───── 2. 카테고리별 샘플 (랜덤) ─────
print("\n" + "=" * 60)
print("[2] 카테고리별 랜덤 샘플 20개")
print("=" * 60)
for cat in ['PER','LOC','OFF','GRP','DATE','SYS','EVT']:
    n = (ck['category']==cat).sum()
    if n == 0:
        continue
    sample = ck[ck['category']==cat]['word'].sample(min(20, n), random_state=42).tolist()
    print(f"\n{cat} ({n}개):")
    print(f"  {sample}")

# ───── 3. NONE 품질 검증 ─────
print("\n" + "=" * 60)
print("[3] NONE 고빈도 TOP 50 — 진짜 NONE인지 검증")
print("=" * 60)
none_words = ck[ck['category']=='NONE']['word'].tolist()
none_freq = (merged[merged['word'].isin(none_words)]
             .groupby('word')['freq'].sum()
             .sort_values(ascending=False)
             .head(50))
print(none_freq.to_string())
print("\n→ 위 목록에 명백한 고유명사(인명/지명/관직)가 있는지 확인")

# ───── 4. 길이별 분포 ─────
print("\n" + "=" * 60)
print("[4] 글자수별 카테고리 분포 (비율)")
print("=" * 60)
ck['len'] = ck['word'].str.len()
print(pd.crosstab(ck['len'], ck['category'], normalize='index').round(2).to_string())
print("\n→ 1글자: NONE 많은 게 정상")
print("→ 3글자+: NONE 50% 미만이면 정상 (대부분 고유명사여야)")

# ───── 5. 지난번 문제 단어 재분류 확인 ─────
print("\n" + "=" * 60)
print("[5] 주요 검증 단어 분류 결과")
print("=" * 60)
check_words = ['大行', '今上', '皇上', '帝', '王貝勒', '王公', '四子',
               '世祖', '聖祖', '宗室', '滿漢', '康熙', '北京', '索尼', '鰲拜']
for w in check_words:
    cat = ck[ck['word']==w]['category'].values
    if len(cat) > 0:
        print(f"  {w:8s} → {cat[0]}")
    else:
        print(f"  {w:8s} → (단어 자체가 데이터에 없음)")

# ───── 6. 분석용 데이터 통계 (NONE/UNK 제외) ─────
print("\n" + "=" * 60)
print("[6] 분석용 데이터 (NONE/UNK 제외)")
print("=" * 60)
analyzable = ck[~ck['category'].isin(['NONE','UNK'])]
print(f"분석 대상 단어 수: {len(analyzable):,}개")
print(f"\n카테고리별:")
print(analyzable['category'].value_counts().to_string())


[1] 전체 카테고리 분포
category
NONE    30232
PER      2688
LOC      1131
OFF       486
SYS       221
DATE      200
GRP       186
EVT        10

UNK 잔존: 0개
전체 단어: 35,154개

[2] 카테고리별 랜덤 샘플 20개

PER (2688개):
  ['達爾瑪', '王毓賢', '圖爾格', '查祿', '蔣國正', '喇木扎', '學士馬', '吳昺', '碩端敏', '王善', '伯噶', '劉漢業', '王盛', '王化致', '郭多', '董宜斯噶卜', '屈盡美', '巡撫布雅', '德赫勒', '岳鍾琪']

LOC (1131개):
  ['保安', '新安', '饒南', '大宛', '烏闌布通', '歷城縣', '尤溪', '廣西', '武陟', '淮', '關外', '金山', '建業', '薩哈連', '大都', '青縣', '西暖閣', '湖州', '黑山', '洛河']

OFF (486개):
  ['司同知', '經略', '平郡王', '河員', '各扎薩克', '汗王', '漢軍學士', '護軍參領', '平西王', '鎮國將', '刑部', '蘇松常道', '左春坊', '陵遣官', '諸佐領', '故大學士', '運同', '巡撫', '今之督撫', '外任']

GRP (186개):
  ['六旗', '西番', '奈曼喀爾', '喀爾喀人', '旗漢', '左翼', '準噶爾賊眾', '左營', '郭爾羅斯扎魯特', '扎魯特喀爾', '提標', '正黃旗漢', '旗丁', '秦喀爾', '倭', '明朝', '科爾沁喀爾', '漢軍漢人', '安科爾沁', '貴州督標']

DATE (200개):
  ['甲寅', '三月', '十七年', '三十三年', '初八', '今春', '乙未', '四十二年', '宋朝', '元旦', '戊辰', '四十七年', '正月初一', '二十二日', '五十四年', '初四', '二年', '二十八年', '壬辰', '丙寅']

SYS (221개):
  ['結案', '三跪九叩', '太學', '科場', '流官', '徭役'

In [6]:
# Cell 5.2 (선택): 트랜잭션 크기 미리보기
print("=" * 60)
print("[월/연도별 트랜잭션 크기 예측]")
print("=" * 60)

# NONE/UNK 제외한 단어만
analyzable_words = set(ck[~ck['category'].isin(['NONE','UNK'])]['word'])
df_filt = merged[merged['word'].isin(analyzable_words)]

# 월별
month_size = df_filt.groupby('period_month')['word'].nunique()
print(f"\n월별 트랜잭션 ({len(month_size)}건):")
print(f"  평균 단어 수: {month_size.mean():.1f}")
print(f"  중앙값:      {month_size.median():.0f}")
print(f"  최소~최대:   {month_size.min()} ~ {month_size.max()}")

# 연도별
year_size = df_filt.groupby('period_year')['word'].nunique()
print(f"\n연도별 트랜잭션 ({len(year_size)}건):")
print(f"  평균 단어 수: {year_size.mean():.1f}")
print(f"  중앙값:      {year_size.median():.0f}")
print(f"  최소~최대:   {year_size.min()} ~ {year_size.max()}")


[월/연도별 트랜잭션 크기 예측]

월별 트랜잭션 (758건):
  평균 단어 수: 27.9
  중앙값:      25
  최소~최대:   4 ~ 124

연도별 트랜잭션 (62건):
  평균 단어 수: 240.3
  중앙값:      242
  최소~최대:   152 ~ 347


In [8]:
# Cell 5.5: 추가 수정 --- 아직 75%라 더 하고 살펴봐야 할 듯 ---
# 해보니까 굉장히 좋아서 하지 않고 진행하기로 함.

In [9]:
# Cell 6: 트랜잭션 CSV 생성
import pandas as pd

INPUT_NER  = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\ner_results.csv"
OUT_MONTH  = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\transactions_month.csv"
OUT_YEAR   = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\transactions_year.csv"

# ───── 옵션 ─────
MIN_FREQ        = 1       # 트랜잭션 크기 적정해서 필터링 불필요
USE_CATEGORIES  = None    # 전체 카테고리. 나중에 ["PER","LOC","OFF","GRP","EVT"] 등으로 좁혀볼 수 있음
# ─────────────────

df = pd.read_csv(INPUT_NER)
print(f"[INFO] NER 결과 로드: {len(df):,}행, 고유 단어 {df['word'].nunique():,}개")
print(f"[INFO] 카테고리 분포:\n{df['category'].value_counts().to_string()}\n")

if USE_CATEGORIES:
    before = len(df)
    df = df[df["category"].isin(USE_CATEGORIES)].reset_index(drop=True)
    print(f"[FILTER] 카테고리 {USE_CATEGORIES}: {before:,} → {len(df):,}행")

if MIN_FREQ > 1:
    before = len(df)
    df = df[df["freq"] >= MIN_FREQ].reset_index(drop=True)
    print(f"[FILTER] freq >= {MIN_FREQ}: {before:,} → {len(df):,}행")


def build_transactions(df, group_col, out_path):
    tx = (
        df.groupby(group_col)["word"]
        .apply(lambda x: sorted(set(x)))
        .reset_index()
    )
    tx["item_count"] = tx["word"].apply(len)
    tx["items"]      = tx["word"].apply(lambda lst: "|".join(lst))
    tx = tx[[group_col, "items", "item_count"]]
    tx.to_csv(out_path, index=False, encoding="utf-8-sig")

    print(f"\n[DONE] {group_col} 트랜잭션 → {out_path}")
    print(f"  트랜잭션 수: {len(tx):,}건")
    print(f"  아이템 수 — 평균: {tx['item_count'].mean():.1f}, "
          f"중앙값: {tx['item_count'].median():.0f}, "
          f"최소: {tx['item_count'].min()}, 최대: {tx['item_count'].max()}")
    return tx


month_tx = build_transactions(df, "period_month", OUT_MONTH)
year_tx  = build_transactions(df, "period_year",  OUT_YEAR)

print(f"\n[월별 샘플]")
print(month_tx.head(3)[["period_month", "item_count"]].to_string(index=False))
print(f"\n[연도별 샘플]")
print(year_tx.head(3)[["period_year", "item_count"]].to_string(index=False))


[INFO] NER 결과 로드: 21,126행, 고유 단어 4,903개
[INFO] 카테고리 분포:
category
OFF     5547
DATE    4761
PER     4406
LOC     4071
SYS     1173
GRP     1139
EVT       29


[DONE] period_month 트랜잭션 → C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\transactions_month.csv
  트랜잭션 수: 758건
  아이템 수 — 평균: 27.9, 중앙값: 25, 최소: 4, 최대: 124

[DONE] period_year 트랜잭션 → C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\transactions_year.csv
  트랜잭션 수: 62건
  아이템 수 — 평균: 240.3, 중앙값: 242, 최소: 152, 최대: 347

[월별 샘플]
period_month  item_count
     1661-01          29
     1661-02          34
     1661-03          60

[연도별 샘플]
 period_year  item_count
        1661         333
        1662         179
        1663         243


In [2]:
# Cell 7: FP-Growth 장바구니 분석 (월별 전용)
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

INPUT_MONTH = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\transactions_month.csv"
INPUT_NER   = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\ner_results.csv"
OUT_MONTH   = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\rules_month.csv"

# ───── 파라미터 ─────
MIN_SUPPORT    = 0.05    # 758개월 중 38개월 이상 등장
MIN_CONFIDENCE = 0.5
MIN_LIFT       = 1.5
MAX_LEN        = 3

# ───── 1. 트랜잭션 로드 ─────
df_m = pd.read_csv(INPUT_MONTH)
transactions = df_m["items"].apply(lambda s: s.split("|")).tolist()
print(f"[INFO] 월별 트랜잭션: {len(transactions):,}건")

# ───── 2. 인코딩 + FP-Growth ─────
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
te_df = pd.DataFrame(te_array, columns=te.columns_)
print(f"[INFO] 고유 아이템: {len(te.columns_):,}개")

freq_items = fpgrowth(te_df, min_support=MIN_SUPPORT, use_colnames=True, max_len=MAX_LEN)
print(f"[INFO] 빈발 아이템셋: {len(freq_items):,}개")

# ───── 3. 연관규칙 ─────
rules = association_rules(freq_items, metric="confidence", min_threshold=MIN_CONFIDENCE)
rules = rules[rules["lift"] >= MIN_LIFT].copy()

rules["antecedents"] = rules["antecedents"].apply(lambda x: ", ".join(sorted(x)))
rules["consequents"] = rules["consequents"].apply(lambda x: ", ".join(sorted(x)))
rules["ant_len"]    = rules["antecedents"].apply(lambda s: len(s.split(", ")))
rules["cons_len"]   = rules["consequents"].apply(lambda s: len(s.split(", ")))
rules = rules.sort_values(["lift","confidence"], ascending=False).reset_index(drop=True)
rules = rules[["antecedents","consequents","ant_len","cons_len","support","confidence","lift"]]

# ───── 4. 카테고리 태그 ─────
ner = pd.read_csv(INPUT_NER)
word2cat = dict(zip(ner["word"], ner["category"]))
def tag(s): return ", ".join(f"{w}({word2cat.get(w,'?')})" for w in s.split(", "))
rules["antecedents_tagged"] = rules["antecedents"].apply(tag)
rules["consequents_tagged"] = rules["consequents"].apply(tag)

# ───── 5. 저장 + 요약 ─────
rules.to_csv(OUT_MONTH, index=False, encoding="utf-8-sig")
print(f"\n[DONE] 월별 룰 {len(rules):,}개 → {OUT_MONTH}")
print(f"\n[상위 15개 (lift 기준)]")
print(rules.head(15)[["antecedents_tagged","consequents_tagged","support","confidence","lift"]].to_string(index=False))


[INFO] 월별 트랜잭션: 758건
[INFO] 고유 아이템: 4,903개
[INFO] 빈발 아이템셋: 673개

[DONE] 월별 룰 1,475개 → C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\rules_month.csv

[상위 15개 (lift 기준)]
antecedents_tagged consequents_tagged  support  confidence      lift
           固山(SYS)            貝子(OFF) 0.051451    0.928571 15.996753
           貝子(OFF)            固山(SYS) 0.051451    0.886364 15.996753
          兼禮部(OFF)   侍郎(OFF), 內閣(OFF) 0.050132    0.950000 15.321277
  侍郎(OFF), 內閣(OFF)           兼禮部(OFF) 0.050132    0.808511 15.321277
           藩王(OFF)   元旦(DATE), 王(OFF) 0.050132    0.678571 13.535714
  元旦(DATE), 王(OFF)            藩王(OFF) 0.050132    1.000000 13.535714
         朝鮮國王(PER)   堂子(LOC), 滿洲(GRP) 0.051451    0.709091 13.437273
  堂子(LOC), 滿洲(GRP)          朝鮮國王(PER) 0.051451    0.975000 13.437273
  八旗(SYS), 堂子(LOC)          朝鮮國王(PER) 0.052770    0.952381 13.125541
         朝鮮國王(PER)   八旗(SYS), 堂子(LOC) 0.052770    0.727273 13.125541
         朝鮮國王(PER)  堂子(LOC), 正月(DATE) 0.071240    0.981818 13.056459
      

In [4]:
# Cell 8: 룰 탐색
import pandas as pd

rules = pd.read_csv(r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\rules_month.csv")


def search_rules(keyword, top=20):
    mask = (rules["antecedents"].str.contains(keyword, na=False) |
            rules["consequents"].str.contains(keyword, na=False))
    result = rules[mask].head(top)
    print(f"\n[검색: '{keyword}'] {mask.sum()}개 / 상위 {len(result)}개")
    cols = ["antecedents_tagged","consequents_tagged","support","confidence","lift"]
    print(result[cols].to_string(index=False))


def filter_by_category(cat_pattern, top=20):
    mask = (rules["antecedents_tagged"].str.contains(cat_pattern, regex=True, na=False) |
            rules["consequents_tagged"].str.contains(cat_pattern, regex=True, na=False))
    result = rules[mask].head(top)
    print(f"\n[카테고리 패턴: {cat_pattern}] {mask.sum()}개 / 상위 {len(result)}개")
    cols = ["antecedents_tagged","consequents_tagged","support","confidence","lift"]
    print(result[cols].to_string(index=False))


# ───── 사용 예시 ─────
# search_rules("噶爾丹")            # 갈단(준가르) 관련
# search_rules("朝鮮")              # 조선 관련
# search_rules("三藩")              # 삼번 관련
# filter_by_category(r"\(EVT\)")    # 사건 포함 룰
# filter_by_category(r"\(PER\).*\(LOC\)")  # 인물+지명 룰

search_rules("朝鮮")              # 조선 관련

filter_by_category(r"\(EVT\)", top=20)



[검색: '朝鮮'] 273개 / 상위 20개
  antecedents_tagged consequents_tagged  support  confidence      lift
           朝鮮國王(PER)   堂子(LOC), 滿洲(GRP) 0.051451    0.709091 13.437273
    堂子(LOC), 滿洲(GRP)          朝鮮國王(PER) 0.051451    0.975000 13.437273
    八旗(SYS), 堂子(LOC)          朝鮮國王(PER) 0.052770    0.952381 13.125541
           朝鮮國王(PER)   八旗(SYS), 堂子(LOC) 0.052770    0.727273 13.125541
           朝鮮國王(PER)  堂子(LOC), 正月(DATE) 0.071240    0.981818 13.056459
           朝鮮國王(PER)  元旦(DATE), 堂子(LOC) 0.071240    0.981818 13.056459
           朝鮮國王(PER)   堂子(LOC), 朔(DATE) 0.071240    0.981818 13.056459
   堂子(LOC), 正月(DATE)          朝鮮國王(PER) 0.071240    0.947368 13.056459
   元旦(DATE), 堂子(LOC)          朝鮮國王(PER) 0.071240    0.947368 13.056459
    堂子(LOC), 朔(DATE)          朝鮮國王(PER) 0.071240    0.947368 13.056459
    堂子(LOC), 藩王(OFF)          朝鮮國王(PER) 0.063325    0.941176 12.971123
   堂子(LOC), 大學士(OFF)          朝鮮國王(PER) 0.063325    0.941176 12.971123
           朝鮮國王(PER)   堂子(LOC), 藩王(OFF) 0.063325   

In [5]:
# 진단: EVT 가 없었음
import pandas as pd
rules = pd.read_csv(r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\rules_month.csv")

# 1. tagged 컬럼 첫 5개 실제 형태 확인 (공백/특수문자 있는지)
print("[1] tagged 컬럼 실제 형태:")
for s in rules["antecedents_tagged"].head(5):
    print(f"  '{s}'  (길이 {len(s)})")

# 2. EVT 문자열이 어디든 들어있는지 단순 검색
mask_simple = (rules["antecedents_tagged"].str.contains("EVT", na=False) |
               rules["consequents_tagged"].str.contains("EVT", na=False))
print(f"\n[2] 'EVT' 단순 포함 룰: {mask_simple.sum()}개")

# 3. EVT 단어 목록
ner = pd.read_csv(r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\ner_results.csv")
evt_words = ner[ner["category"]=="EVT"]["word"].unique()
print(f"\n[3] EVT 단어 ({len(evt_words)}개): {list(evt_words)}")

# 4. EVT 단어가 월별 트랜잭션에 몇 번 등장했는지
tx = pd.read_csv(r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\transactions_month.csv")
print(f"\n[4] EVT 단어별 등장 월 수 (758개월 중):")
for w in evt_words:
    count = tx["items"].str.contains(w, na=False).sum()
    pct = count / len(tx) * 100
    print(f"  {w}: {count}개월 ({pct:.1f}%)  → support 임계값(5%, 38개월) {'통과' if pct>=5 else '미달'}")


[1] tagged 컬럼 실제 형태:
  '固山(SYS)'  (길이 7)
  '貝子(OFF)'  (길이 7)
  '兼禮部(OFF)'  (길이 8)
  '侍郎(OFF), 內閣(OFF)'  (길이 16)
  '藩王(OFF)'  (길이 7)

[2] 'EVT' 단순 포함 룰: 0개

[3] EVT 단어 (10개): ['南巡', '喀爾喀會盟', '靖難', '喀爾喀降', '平噶爾丹', '征噶爾丹', '耿逆之變', '三逆', '征拉藏', '平三逆']

[4] EVT 단어별 등장 월 수 (758개월 중):
  南巡: 19개월 (2.5%)  → support 임계값(5%, 38개월) 미달
  喀爾喀會盟: 2개월 (0.3%)  → support 임계값(5%, 38개월) 미달
  靖難: 1개월 (0.1%)  → support 임계값(5%, 38개월) 미달
  喀爾喀降: 1개월 (0.1%)  → support 임계값(5%, 38개월) 미달
  平噶爾丹: 1개월 (0.1%)  → support 임계값(5%, 38개월) 미달
  征噶爾丹: 1개월 (0.1%)  → support 임계값(5%, 38개월) 미달
  耿逆之變: 1개월 (0.1%)  → support 임계값(5%, 38개월) 미달
  三逆: 2개월 (0.3%)  → support 임계값(5%, 38개월) 미달
  征拉藏: 1개월 (0.1%)  → support 임계값(5%, 38개월) 미달
  平三逆: 1개월 (0.1%)  → support 임계값(5%, 38개월) 미달


In [ ]:
# 해결 처방전: EVT 전용 분석 (임계값 대폭 완화 --- 미달이었음 다)
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

INPUT_MONTH = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\transactions_month.csv"
INPUT_NER   = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\ner_results.csv"
OUT_EVT     = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\rules_month_evt.csv"

# EVT 단어 목록
ner = pd.read_csv(INPUT_NER)
evt_words = set(ner[ner["category"]=="EVT"]["word"])
print(f"[INFO] EVT 단어: {evt_words}")

# 트랜잭션 로드
df_m = pd.read_csv(INPUT_MONTH)
transactions = df_m["items"].apply(lambda s: s.split("|")).tolist()

# EVT 단어가 하나라도 들어간 월만 골라서 분석
evt_tx = [tx for tx in transactions if any(w in evt_words for w in tx)]
print(f"[INFO] EVT 등장 월: {len(evt_tx)}개월")

# 임계값을 매우 낮게
te = TransactionEncoder()
te_df = pd.DataFrame(te.fit(evt_tx).transform(evt_tx), columns=te.columns_)
freq = fpgrowth(te_df, min_support=0.15, use_colnames=True, max_len=3)
rules = association_rules(freq, metric="confidence", min_threshold=0.5)
rules = rules[rules["lift"] >= 2.0].copy()

# EVT가 포함된 룰만 필터링
def has_evt(itemset):
    return any(w in evt_words for w in itemset)
rules = rules[rules["antecedents"].apply(has_evt) | rules["consequents"].apply(has_evt)]

# 정리
rules["antecedents"] = rules["antecedents"].apply(lambda x: ", ".join(sorted(x)))
rules["consequents"] = rules["consequents"].apply(lambda x: ", ".join(sorted(x)))
rules = rules.sort_values("lift", ascending=False).reset_index(drop=True)

# 카테고리 태그
word2cat = dict(zip(ner["word"], ner["category"]))
def tag(s): return ", ".join(f"{w}({word2cat.get(w,'?')})" for w in s.split(", "))
rules["antecedents_tagged"] = rules["antecedents"].apply(tag)
rules["consequents_tagged"] = rules["consequents"].apply(tag)

print(f"\n[DONE] EVT 룰: {len(rules)}개")
print(rules.head(20)[["antecedents_tagged","consequents_tagged","support","confidence","lift"]].to_string(index=False))

rules.to_csv(OUT_EVT, index=False, encoding="utf-8-sig")


In [5]:
# Cell 9: 시각화 5종 PNG 저장
import os
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import networkx as nx
from collections import Counter
from ast import literal_eval

# ───── 경로 ─────
CSV_DIR    = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv"
OUT_DIR    = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\figures"
os.makedirs(OUT_DIR, exist_ok=True)

INPUT_NER   = os.path.join(CSV_DIR, "ner_results.csv")
INPUT_RULES = os.path.join(CSV_DIR, "rules_month.csv")
INPUT_MONTH = os.path.join(CSV_DIR, "transactions_month.csv")

# ───── 한자 지원 폰트 자동 탐색 ─────
def find_cjk_font():
    candidates = ["Malgun Gothic", "SimSun", "Microsoft YaHei", "SimHei",
                  "NanumGothic", "Batang", "Gulim"]
    available = {f.name for f in fm.fontManager.ttflist}
    for c in candidates:
        if c in available:
            print(f"[FONT] 사용 폰트: {c}")
            return c
    print("[WARN] CJK 폰트 못 찾음. 한자가 깨질 수 있습니다.")
    return None

CJK_FONT = find_cjk_font()
if CJK_FONT:
    plt.rcParams["font.family"] = CJK_FONT
plt.rcParams["axes.unicode_minus"] = False  # 마이너스 기호 깨짐 방지
plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.dpi"] = 200
plt.rcParams["savefig.bbox"] = "tight"

# ───── 데이터 로드 ─────
ner   = pd.read_csv(INPUT_NER)
rules = pd.read_csv(INPUT_RULES)
tx    = pd.read_csv(INPUT_MONTH)
word2cat = dict(zip(ner["word"], ner["category"]))

# 카테고리별 색상 통일
CAT_COLORS = {
    "PER": "#E63946",   # 빨강
    "LOC": "#2A9D8F",   # 청록
    "OFF": "#264653",   # 짙은 청
    "GRP": "#F4A261",   # 주황
    "DATE": "#9D4EDD",  # 보라
    "SYS": "#6A994E",   # 녹색
    "EVT": "#000000",   # 검정 (강조)
    "?":   "#999999",
}

print(f"[INFO] 룰 {len(rules):,}개, 트랜잭션 {len(tx):,}건, NER {len(ner):,}행")
print(f"[INFO] 출력 폴더: {OUT_DIR}\n")


# ════════════════════════════════════════════════════════════
# (1) 네트워크 그래프 — 상위 룰 시각화
# ════════════════════════════════════════════════════════════
print("[1/5] 네트워크 그래프 생성 중...")

# 1-항 → 1-항 룰만 사용 (네트워크 가독성)
single_rules = rules[(rules["ant_len"]==1) & (rules["cons_len"]==1)].copy()
# 양방향 중복 제거 (A→B와 B→A 중 lift 큰 것만)
single_rules["pair"] = single_rules.apply(
    lambda r: tuple(sorted([r["antecedents"], r["consequents"]])), axis=1
)
single_rules = single_rules.sort_values("lift", ascending=False).drop_duplicates("pair")

# 상위 80개 룰만 (가독성)
top_rules = single_rules.head(80)

G = nx.Graph()
for _, r in top_rules.iterrows():
    a, b = r["antecedents"], r["consequents"]
    G.add_edge(a, b, weight=r["lift"], support=r["support"])

# 노드 카테고리
node_colors = [CAT_COLORS.get(word2cat.get(n, "?"), "#999999") for n in G.nodes()]
node_sizes  = [300 + G.degree(n)*150 for n in G.nodes()]
edge_widths = [G[u][v]["weight"]*0.3 for u, v in G.edges()]

fig, ax = plt.subplots(figsize=(16, 12))
pos = nx.spring_layout(G, k=1.2, iterations=80, seed=42)

nx.draw_networkx_edges(G, pos, width=edge_widths, alpha=0.4, edge_color="#888888", ax=ax)
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes,
                       alpha=0.85, edgecolors="white", linewidths=1.5, ax=ax)
nx.draw_networkx_labels(G, pos, font_family=CJK_FONT, font_size=10,
                        font_weight="bold", ax=ax)

# 범례
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=cat) for cat, c in CAT_COLORS.items() if cat != "?"]
ax.legend(handles=legend_elements, loc="upper left", fontsize=11,
          title="카테고리", title_fontsize=12)

ax.set_title("청성조실록 한문 단어 동시 출현 네트워크 (상위 80개 룰)",
             fontsize=16, fontweight="bold", pad=20)
ax.axis("off")
plt.savefig(os.path.join(OUT_DIR, "01_network.png"))
plt.close()
print(f"  → 01_network.png")


# ════════════════════════════════════════════════════════════
# (2) 산점도 — Support × Lift × Confidence
# ════════════════════════════════════════════════════════════
print("[2/5] Support×Lift 산점도 생성 중...")

fig, ax = plt.subplots(figsize=(12, 8))
scatter = ax.scatter(rules["support"], rules["lift"],
                     c=rules["confidence"], cmap="viridis",
                     s=40, alpha=0.6, edgecolors="white", linewidths=0.3)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label("Confidence", fontsize=12)

# 강력한 룰 (lift > 10) 텍스트 라벨
strong = rules[rules["lift"] > 10].head(15)
for _, r in strong.iterrows():
    label = f"{r['antecedents'][:8]}→{r['consequents'][:8]}"
    ax.annotate(label, (r["support"], r["lift"]),
                fontsize=8, alpha=0.7, xytext=(5, 5), textcoords="offset points")

ax.axhline(y=1, color="red", linestyle="--", alpha=0.5, label="Lift = 1 (우연 수준)")
ax.set_xlabel("Support (지지도)", fontsize=13)
ax.set_ylabel("Lift (향상도)", fontsize=13)
ax.set_title(f"룰 분포 — Support × Lift × Confidence  (전체 {len(rules):,}개)",
             fontsize=15, fontweight="bold", pad=15)
ax.legend(loc="upper right")
ax.grid(alpha=0.3)
plt.savefig(os.path.join(OUT_DIR, "02_scatter.png"))
plt.close()
print(f"  → 02_scatter.png")


# ════════════════════════════════════════════════════════════
# (3) 카테고리 조합 히트맵
# ════════════════════════════════════════════════════════════
print("[3/5] 카테고리 히트맵 생성 중...")

# 각 룰의 antecedent/consequent 카테고리 추출
def get_cats(items_str):
    return [word2cat.get(w.strip(), "?") for w in items_str.split(",")]

heatmap_data = Counter()
for _, r in rules.iterrows():
    ant_cats = set(get_cats(r["antecedents"]))
    con_cats = set(get_cats(r["consequents"]))
    for a in ant_cats:
        for c in con_cats:
            pair = tuple(sorted([a, c]))
            heatmap_data[pair] += 1

cats = ["PER", "LOC", "OFF", "GRP", "DATE", "SYS", "EVT"]
matrix = np.zeros((len(cats), len(cats)))
for (a, b), v in heatmap_data.items():
    if a in cats and b in cats:
        i, j = cats.index(a), cats.index(b)
        matrix[i][j] = v
        if i != j:
            matrix[j][i] = v

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(matrix, annot=True, fmt=".0f", cmap="YlOrRd",
            xticklabels=cats, yticklabels=cats,
            cbar_kws={"label": "룰 개수"}, ax=ax,
            linewidths=0.5, linecolor="white")
ax.set_title("카테고리 조합별 룰 개수 (대칭 행렬)",
             fontsize=15, fontweight="bold", pad=15)
ax.set_xlabel("카테고리 B", fontsize=12)
ax.set_ylabel("카테고리 A", fontsize=12)
plt.savefig(os.path.join(OUT_DIR, "03_heatmap.png"))
plt.close()
print(f"  → 03_heatmap.png")


# ════════════════════════════════════════════════════════════
# (4) 시계열 — 핵심 룰 발동 패턴
# ════════════════════════════════════════════════════════════
print("[4/5] 시계열 그래프 생성 중...")

# 핵심 룰 5개 정의 (실제 결과에서 가져옴)
target_rules = [
    ("朝鮮國王 + 堂子",  ["朝鮮國王", "堂子"]),
    ("固山 + 貝子",     ["固山", "貝子"]),
    ("元旦 + 藩王",     ["元旦", "藩王"]),
    ("八旗 + 滿洲",     ["八旗", "滿洲"]),
    ("大學士 + 內閣",   ["大學士", "內閣"]),
]

fig, axes = plt.subplots(len(target_rules), 1, figsize=(14, 10), sharex=True)

# 트랜잭션을 dict로 변환
tx_dict = {}
for _, row in tx.iterrows():
    pm = row["period_month"]
    items = set(row["items"].split("|"))
    tx_dict[pm] = items

# 월 → 연도 변환
def month_to_year(pm):
    return int(pm.split("-")[0])

all_months = sorted(tx_dict.keys())
years = [month_to_year(m) for m in all_months]

for ax, (label, words) in zip(axes, target_rules):
    activations = [1 if all(w in tx_dict[m] for w in words) else 0 for m in all_months]
    # 연도별 합계
    year_count = Counter()
    for y, a in zip(years, activations):
        year_count[y] += a
    xs = sorted(year_count.keys())
    ys = [year_count[y] for y in xs]
    ax.bar(xs, ys, color="#264653", alpha=0.8, width=0.8)
    ax.set_ylabel(label, fontsize=11, rotation=0, ha="right", va="center", labelpad=50)
    ax.set_ylim(0, max(ys)+1 if ys else 1)
    ax.grid(alpha=0.3, axis="y")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

axes[-1].set_xlabel("연도", fontsize=12)
fig.suptitle("핵심 룰의 연도별 발동 횟수 (강희 1661-1722)",
             fontsize=15, fontweight="bold", y=0.995)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "04_timeline.png"))
plt.close()
print(f"  → 04_timeline.png")


# ════════════════════════════════════════════════════════════
# (5) 카테고리별 단어 분포 막대그래프
# ════════════════════════════════════════════════════════════
print("[5/5] 카테고리 분포 막대그래프 생성 중...")

# 단어 기준
word_dist = ner.drop_duplicates("word")["category"].value_counts()
# 행 기준 (분석 데이터)
row_dist  = ner["category"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, data, title in [
    (axes[0], word_dist, "고유 단어 수 (단어 기준)"),
    (axes[1], row_dist,  "출현 횟수 (행 기준)")
]:
    cats_ordered = [c for c in ["PER","LOC","OFF","GRP","DATE","SYS","EVT"] if c in data.index]
    values = [data[c] for c in cats_ordered]
    colors = [CAT_COLORS[c] for c in cats_ordered]
    bars = ax.bar(cats_ordered, values, color=colors, edgecolor="white", linewidth=1.5)
    for bar, v in zip(bars, values):
        ax.text(bar.get_x()+bar.get_width()/2, v, f"{v:,}",
                ha="center", va="bottom", fontsize=11, fontweight="bold")
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_ylabel("개수", fontsize=11)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(alpha=0.3, axis="y")

fig.suptitle("NER 카테고리 분포 (NONE 제외)",
             fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "05_categories.png"))
plt.close()
print(f"  → 05_categories.png")


# ════════════════════════════════════════════════════════════
# 완료
# ════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print(f"[DONE] 5개 PNG 저장 완료 → {OUT_DIR}")
print(f"{'='*60}")
for f in sorted(os.listdir(OUT_DIR)):
    fpath = os.path.join(OUT_DIR, f)
    size_kb = os.path.getsize(fpath) / 1024
    print(f"  {f}  ({size_kb:.0f} KB)")


[FONT] 사용 폰트: Malgun Gothic
[INFO] 룰 1,475개, 트랜잭션 758건, NER 21,126행
[INFO] 출력 폴더: C:\Users\USER\OneDrive\바탕 화면\nerproject\figures

[1/5] 네트워크 그래프 생성 중...


C:\Users\USER\AppData\Local\Temp\ipykernel_15164\269155935.py:108: UserWarning: Glyph 37070 (\N{CJK UNIFIED IDEOGRAPH-90CE}) missing from font(s) Malgun Gothic.
  plt.savefig(os.path.join(OUT_DIR, "01_network.png"))
C:\Users\USER\AppData\Local\Temp\ipykernel_15164\269155935.py:108: UserWarning: Glyph 23578 (\N{CJK UNIFIED IDEOGRAPH-5C1A}) missing from font(s) Malgun Gothic.
  plt.savefig(os.path.join(OUT_DIR, "01_network.png"))


  → 01_network.png
[2/5] Support×Lift 산점도 생성 중...


C:\Users\USER\AppData\Local\Temp\ipykernel_15164\269155935.py:139: UserWarning: Glyph 37070 (\N{CJK UNIFIED IDEOGRAPH-90CE}) missing from font(s) Malgun Gothic.
  plt.savefig(os.path.join(OUT_DIR, "02_scatter.png"))


  → 02_scatter.png
[3/5] 카테고리 히트맵 생성 중...
  → 03_heatmap.png
[4/5] 시계열 그래프 생성 중...
  → 04_timeline.png
[5/5] 카테고리 분포 막대그래프 생성 중...
  → 05_categories.png

[DONE] 5개 PNG 저장 완료 → C:\Users\USER\OneDrive\바탕 화면\nerproject\figures
  01_network.png  (296 KB)
  02_scatter.png  (295 KB)
  03_heatmap.png  (95 KB)
  04_timeline.png  (86 KB)
  05_categories.png  (86 KB)


In [2]:
# Cell 10: 朝鮮國王 중심 심층 분석 (3종)
import os
import math
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from collections import Counter

# ───── 경로 ─────
CSV_DIR  = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv"
OUT_DIR  = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\figures"
os.makedirs(OUT_DIR, exist_ok=True)

INPUT_NER     = os.path.join(CSV_DIR, "ner_results.csv")
INPUT_NER_ALL = os.path.join(CSV_DIR, "ner_results_all.csv")  # NONE 포함 전체
INPUT_TX      = os.path.join(CSV_DIR, "transactions_month.csv")
INPUT_MERGED  = os.path.join(CSV_DIR, "merged.csv")

# ───── 폰트 ─────
def find_cjk_font():
    candidates = ["Malgun Gothic", "SimSun", "Microsoft YaHei", "SimHei", "Batang"]
    available = {f.name for f in fm.fontManager.ttflist}
    for c in candidates:
        if c in available:
            return c
    return None

CJK_FONT = find_cjk_font()
if CJK_FONT:
    plt.rcParams["font.family"] = CJK_FONT
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["savefig.dpi"] = 200
plt.rcParams["savefig.bbox"] = "tight"

# ───── 데이터 로드 ─────
df_ner     = pd.read_csv(INPUT_NER)
df_ner_all = pd.read_csv(INPUT_NER_ALL)
tx         = pd.read_csv(INPUT_TX)
merged     = pd.read_csv(INPUT_MERGED)

word2cat = dict(zip(df_ner_all["word"], df_ner_all["category"]))

TARGET = "朝鮮國王"

# 월 추출 함수
def extract_month(pm):
    parts = pm.split("-")
    if "yun" in parts[1]:
        return int(parts[1].replace("yun","")) + 100
    return int(parts[1])

print(f"[INFO] 분석 대상: '{TARGET}'")
print(f"[INFO] 출력 폴더: {OUT_DIR}\n")

# ════════════════════════════════════════════════════════════════════
# 분석 1: 朝鮮國王 ↔ 날짜 단어 관계 심층 분석
# ════════════════════════════════════════════════════════════════════
print("="*70)
print("[분석 1] 朝鮮國王과 날짜 단어 관계")
print("="*70)

# 1-A. 朝鮮國王이 등장한 월의 월(month) 분포
mask = tx["items"].str.contains(TARGET, na=False)
appeared = tx[mask]["period_month"].tolist()
print(f"\n  朝鮮國王 등장 월: {len(appeared)}/{len(tx)} ({len(appeared)/len(tx):.1%})")

months_only = [extract_month(m) for m in appeared if extract_month(m) <= 12]
yun_months  = [extract_month(m) - 100 for m in appeared if extract_month(m) > 100]
month_dist = Counter(months_only)

print(f"\n  [월별 분포 (일반 달)]")
for m in range(1, 13):
    bar = "█" * month_dist.get(m, 0)
    print(f"    {m:2d}월: {month_dist.get(m,0):3d}회  {bar}")
if yun_months:
    print(f"  [윤달] {Counter(yun_months)}")

jan_count = month_dist.get(1, 0)
print(f"\n  → 정월(1월) 집중도: {jan_count}/{len(appeared)} = {jan_count/len(appeared):.1%}")

# 1-B. 朝鮮國王이 등장한 월에 함께 등장한 DATE 단어 빈도
date_words = set(df_ner[df_ner["category"]=="DATE"]["word"])
date_cooccur = Counter()
for items in tx[mask]["items"]:
    for w in set(items.split("|")) & date_words:
        date_cooccur[w] += 1

print(f"\n  [朝鮮國王 등장 월에 동반 등장한 DATE 단어 TOP 15]")
top_date = date_cooccur.most_common(15)
for w, c in top_date:
    pct = c / len(appeared)
    print(f"    {w:8s}: {c}/{len(appeared)} ({pct:.1%})")

# ════════════════════════════════════════════════════════════════════
# 분석 2: 朝鮮國王 ↔ 堂子 비율 분석
# ════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("[분석 2] 朝鮮國王 ↔ 堂子 동시 출현 비율")
print("="*70)

PARTNER = "堂子"
mask_t = tx["items"].str.contains(TARGET,  na=False)
mask_p = tx["items"].str.contains(PARTNER, na=False)

n_total      = len(tx)
n_target     = mask_t.sum()
n_partner    = mask_p.sum()
n_both       = (mask_t & mask_p).sum()
n_target_no  = (mask_t & ~mask_p).sum()
n_partner_no = (~mask_t & mask_p).sum()

print(f"\n  전체 월수             : {n_total}")
print(f"  朝鮮國王 등장          : {n_target}")
print(f"  堂子 등장              : {n_partner}")
print(f"  둘 다 등장             : {n_both}")
print(f"\n  P(堂子 | 朝鮮國王)    : {n_both/n_target:.1%}  ← 조선국왕 등장 시 당자 동반율")
print(f"  P(朝鮮國王 | 堂子)    : {n_both/n_partner:.1%}  ← 당자 등장 시 조선국왕 동반율")
print(f"\n  비대칭 해석:")
print(f"    조선국왕은 거의 항상 당자와 함께 ({n_both/n_target:.1%})")
print(f"    그러나 당자는 조선국왕 외에도 등장 ({1-n_both/n_partner:.1%}는 조선국왕 없이)")

# Fisher's exact test (통계적 유의성)
from scipy.stats import fisher_exact
table = [[n_both, n_target_no],
         [n_partner_no, n_total - n_target - n_partner + n_both]]
odds, pval = fisher_exact(table)
print(f"\n  Fisher's exact test:")
print(f"    Odds ratio: {odds:.2f}")
print(f"    p-value   : {pval:.2e}")

# ════════════════════════════════════════════════════════════════════
# 분석 3: 朝鮮國王 ↔ NONE 단어 동시 출현 강도
# ════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("[분석 3] 朝鮮國王과 NONE 단어들의 동시 출현 강도")
print("="*70)

# NONE 단어 목록
none_words = set(df_ner_all[df_ner_all["category"]=="NONE"]["word"])
print(f"\n  NONE 단어 총수: {len(none_words):,}개")

# transactions_month는 NONE 제외 데이터로 만들어졌으므로,
# NONE 포함 트랜잭션을 merged.csv로부터 재구성해야 함
print("  [INFO] NONE 포함 트랜잭션 재구성 중...")
month_word_map = {}
for pm, group in merged.groupby("period_month"):
    month_word_map[pm] = set(group["word"].astype(str))

# 朝鮮國王이 등장한 월 (merged 기준 재확인)
target_months = {pm for pm, ws in month_word_map.items() if TARGET in ws}
print(f"  朝鮮國王 등장 월(merged 기준): {len(target_months)}")

# 각 NONE 단어의 동시 출현 강도 계산
results = []
n_T = len(target_months)
n_total_m = len(month_word_map)

for w in none_words:
    if w == TARGET:
        continue
    w_months = {pm for pm, ws in month_word_map.items() if w in ws}
    n_W = len(w_months)
    n_TW = len(target_months & w_months)
    if n_W < 5 or n_TW < 3:  # 너무 희귀한 건 제외
        continue
    
    # Jaccard
    jaccard = n_TW / len(target_months | w_months)
    # P(W|T)
    p_w_given_t = n_TW / n_T
    # PMI
    p_t   = n_T / n_total_m
    p_w   = n_W / n_total_m
    p_tw  = n_TW / n_total_m
    pmi   = math.log2(p_tw / (p_t * p_w)) if p_tw > 0 else float("-inf")
    # Lift
    lift  = (n_TW * n_total_m) / (n_T * n_W) if n_T*n_W > 0 else 0
    
    results.append({
        "word": w,
        "n_W": n_W,
        "n_TW": n_TW,
        "P(W|T)": p_w_given_t,
        "Jaccard": jaccard,
        "PMI": pmi,
        "Lift": lift,
    })

none_df = pd.DataFrame(results)
print(f"  분석 가능 NONE 단어: {len(none_df):,}개 (n_W≥5 & n_TW≥3)")

# Lift 기준 상위
print(f"\n  [Lift 기준 상위 20개 NONE 단어 — 朝鮮國王과 강하게 결합]")
top_lift = none_df.sort_values("Lift", ascending=False).head(20)
print(top_lift[["word","n_W","n_TW","P(W|T)","Jaccard","PMI","Lift"]].to_string(index=False))

# 저장
none_df.sort_values("Lift", ascending=False).to_csv(
    os.path.join(CSV_DIR, "joseon_none_cooccurrence.csv"),
    index=False, encoding="utf-8-sig"
)


# ════════════════════════════════════════════════════════════════════
# 시각화 4종
# ════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("[시각화] PNG 4종 저장 시작")
print("="*70)

# ───────────────────────────────────────────────
# 시각화 1: 月별 등장 분포 (분석 1)
# ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 좌: 月별 막대그래프
months_x = list(range(1, 13))
counts   = [month_dist.get(m, 0) for m in months_x]
colors   = ["#E63946" if m == 1 else "#264653" for m in months_x]
axes[0].bar(months_x, counts, color=colors, edgecolor="white", linewidth=1.5)
for i, c in enumerate(counts):
    if c > 0:
        axes[0].text(i+1, c, str(c), ha="center", va="bottom", fontsize=11, fontweight="bold")
axes[0].set_xticks(months_x)
axes[0].set_xlabel("月", fontsize=12)
axes[0].set_ylabel("등장 횟수", fontsize=12)
axes[0].set_title(f"'{TARGET}' 의 月별 등장 분포\n(정월 집중도: {jan_count/len(appeared):.1%})",
                  fontsize=13, fontweight="bold")
axes[0].grid(alpha=0.3, axis="y")

# 우: 동반 DATE 단어 (전체 비율)
words_d  = [w for w, _ in top_date[:12]]
counts_d = [c for _, c in top_date[:12]]
pcts_d   = [c/len(appeared)*100 for c in counts_d]
y_pos = np.arange(len(words_d))
bars = axes[1].barh(y_pos, pcts_d, color="#9D4EDD", edgecolor="white", linewidth=1.5)
for i, (c, p) in enumerate(zip(counts_d, pcts_d)):
    axes[1].text(p+1, i, f"{c}회 ({p:.0f}%)", va="center", fontsize=10)
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels(words_d, fontsize=11)
axes[1].invert_yaxis()
axes[1].set_xlabel(f"朝鮮國王 등장 월 중 동반 비율 (%)", fontsize=12)
axes[1].set_title(f"'{TARGET}' 와 동반 등장한 DATE 단어 TOP 12",
                  fontsize=13, fontweight="bold")
axes[1].grid(alpha=0.3, axis="x")

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "06_joseon_date.png"))
plt.close()
print("  → 06_joseon_date.png")

# ───────────────────────────────────────────────
# 시각화 2: 朝鮮國王 ↔ 堂子 벤 다이어그램 + 비율
# ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 좌: 벤 다이어그램 풍 시각화 (matplotlib 기본 도형)
from matplotlib.patches import Circle
ax = axes[0]
c1 = Circle((0.35, 0.5), 0.25, alpha=0.5, color="#E63946", label="朝鮮國王")
c2 = Circle((0.6, 0.5),  0.30, alpha=0.5, color="#2A9D8F", label="堂子")
ax.add_patch(c1)
ax.add_patch(c2)
ax.text(0.20, 0.5, f"{n_target-n_both}",   ha="center", va="center", fontsize=18, fontweight="bold")
ax.text(0.48, 0.5, f"{n_both}",            ha="center", va="center", fontsize=22, fontweight="bold", color="white")
ax.text(0.78, 0.5, f"{n_partner-n_both}",  ha="center", va="center", fontsize=18, fontweight="bold")
ax.text(0.20, 0.85, "朝鮮國王",  ha="center", fontsize=14, fontweight="bold", color="#E63946")
ax.text(0.78, 0.85, "堂子",      ha="center", fontsize=14, fontweight="bold", color="#2A9D8F")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title(f"동시 출현 분포 (전체 {n_total}개월)",
             fontsize=13, fontweight="bold")

# 우: 조건부 확률 비교 막대
ax = axes[1]
labels = ["P(堂子 | 朝鮮國王)", "P(朝鮮國王 | 堂子)", "P(朝鮮國王)\n[기저율]", "P(堂子)\n[기저율]"]
values = [n_both/n_target, n_both/n_partner, n_target/n_total, n_partner/n_total]
colors_b = ["#E63946", "#2A9D8F", "#cccccc", "#cccccc"]
bars = ax.bar(labels, [v*100 for v in values], color=colors_b, edgecolor="white", linewidth=1.5)
for bar, v in zip(bars, values):
    ax.text(bar.get_x()+bar.get_width()/2, v*100+1.5, f"{v:.1%}",
            ha="center", va="bottom", fontsize=12, fontweight="bold")
ax.set_ylabel("확률 (%)", fontsize=12)
ax.set_ylim(0, 110)
ax.set_title(f"조건부 확률 비교\nFisher p={pval:.2e}",
             fontsize=13, fontweight="bold")
ax.grid(alpha=0.3, axis="y")
plt.setp(ax.get_xticklabels(), fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "07_joseon_dangja.png"))
plt.close()
print("  → 07_joseon_dangja.png")

# ───────────────────────────────────────────────
# 시각화 3: 朝鮮國王 ↔ NONE 단어 산점도 (PMI × P(W|T))
# ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# 좌: 산점도 (PMI × n_TW)
ax = axes[0]
sc = ax.scatter(none_df["PMI"], none_df["P(W|T)"]*100,
                s=none_df["n_TW"]*8, c=none_df["Lift"], 
                cmap="viridis", alpha=0.6, edgecolors="white", linewidths=0.5)
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label("Lift", fontsize=11)
# 상위 12개 라벨
top12 = none_df.sort_values("Lift", ascending=False).head(12)
for _, r in top12.iterrows():
    ax.annotate(r["word"], (r["PMI"], r["P(W|T)"]*100),
                fontsize=10, alpha=0.85, xytext=(4,4), textcoords="offset points",
                fontweight="bold")
ax.set_xlabel("PMI (Pointwise Mutual Information)", fontsize=12)
ax.set_ylabel("P(NONE 단어 | 朝鮮國王 등장) (%)", fontsize=12)
ax.set_title("朝鮮國王과 NONE 단어의 동시 출현 강도\n(점 크기 = 동시 등장 월수, 색 = Lift)",
             fontsize=13, fontweight="bold")
ax.grid(alpha=0.3)

# 우: Lift 상위 20개 막대
ax = axes[1]
top20 = none_df.sort_values("Lift", ascending=False).head(20)
y_pos = np.arange(len(top20))
bars = ax.barh(y_pos, top20["Lift"], color="#F4A261", edgecolor="white", linewidth=1.5)
for i, (_, r) in enumerate(top20.iterrows()):
    ax.text(r["Lift"]+0.1, i, f"L={r['Lift']:.1f} | {r['n_TW']}/{r['n_W']}월",
            va="center", fontsize=9)
ax.set_yticks(y_pos)
ax.set_yticklabels(top20["word"], fontsize=10)
ax.invert_yaxis()
ax.set_xlabel("Lift (우연 대비 연관 강도)", fontsize=12)
ax.set_title("朝鮮國王과 강한 연관 NONE 단어 TOP 20",
             fontsize=13, fontweight="bold")
ax.axvline(x=1, color="red", linestyle="--", alpha=0.5)
ax.grid(alpha=0.3, axis="x")

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "08_joseon_none.png"))
plt.close()
print("  → 08_joseon_none.png")

# ───────────────────────────────────────────────
# 시각화 4: 종합 — 카테고리별 동반 단어 분포
# ───────────────────────────────────────────────
# 朝鮮國王이 등장한 월에 같이 등장한 모든 단어를 카테고리별로 집계
all_cooccur_cats = Counter()
for pm in target_months:
    for w in month_word_map[pm]:
        if w == TARGET:
            continue
        cat = word2cat.get(w, "?")
        all_cooccur_cats[cat] += 1

fig, ax = plt.subplots(figsize=(11, 6))
cats_order = ["NONE", "PER", "LOC", "OFF", "GRP", "DATE", "SYS", "EVT"]
cats_present = [c for c in cats_order if c in all_cooccur_cats]
counts = [all_cooccur_cats[c] for c in cats_present]
colors_c = {"PER":"#E63946","LOC":"#2A9D8F","OFF":"#264653",
            "GRP":"#F4A261","DATE":"#9D4EDD","SYS":"#6A994E",
            "EVT":"#000000","NONE":"#999999"}
bar_colors = [colors_c.get(c,"#999999") for c in cats_present]
bars = ax.bar(cats_present, counts, color=bar_colors, edgecolor="white", linewidth=1.5)
total = sum(counts)
for bar, c in zip(bars, counts):
    ax.text(bar.get_x()+bar.get_width()/2, c, f"{c:,}\n({c/total:.0%})",
            ha="center", va="bottom", fontsize=10, fontweight="bold")
ax.set_ylabel("동반 등장 단어 수 (중복 포함)", fontsize=12)
ax.set_title(f"朝鮮國王 등장 월에 동반 등장한 단어의 카테고리 분포\n"
             f"(전체 {len(target_months)}개월, 중복 포함 총 {total:,}건)",
             fontsize=13, fontweight="bold")
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "09_joseon_category_dist.png"))
plt.close()
print("  → 09_joseon_category_dist.png")

print(f"\n{'='*70}")
print(f"[DONE] 분석 완료")
print(f"{'='*70}")
print(f"  CSV: {os.path.join(CSV_DIR, 'joseon_none_cooccurrence.csv')}")
print(f"  PNG: {OUT_DIR} 폴더")
print(f"    - 06_joseon_date.png        (분석 1)")
print(f"    - 07_joseon_dangja.png      (분석 2)")
print(f"    - 08_joseon_none.png        (분석 3)")
print(f"    - 09_joseon_category_dist.png (종합)")


[INFO] 분석 대상: '朝鮮國王'
[INFO] 출력 폴더: C:\Users\USER\OneDrive\바탕 화면\nerproject\figures

[분석 1] 朝鮮國王과 날짜 단어 관계

  朝鮮國王 등장 월: 55/758 (7.3%)

  [월별 분포 (일반 달)]
     1월:  55회  ███████████████████████████████████████████████████████
     2월:   0회  
     3월:   0회  
     4월:   0회  
     5월:   0회  
     6월:   0회  
     7월:   0회  
     8월:   0회  
     9월:   0회  
    10월:   0회  
    11월:   0회  
    12월:   0회  

  → 정월(1월) 집중도: 55/55 = 100.0%

  [朝鮮國王 등장 월에 동반 등장한 DATE 단어 TOP 15]
    正月      : 55/55 (100.0%)
    康熙      : 55/55 (100.0%)
    元旦      : 55/55 (100.0%)
    朔       : 54/55 (98.2%)
    丁卯      : 7/55 (12.7%)
    乙卯      : 6/55 (10.9%)
    辛未      : 6/55 (10.9%)
    庚午      : 5/55 (9.1%)
    丙戌      : 5/55 (9.1%)
    庚子      : 5/55 (9.1%)
    戊戌      : 5/55 (9.1%)
    壬寅      : 5/55 (9.1%)
    丁亥      : 5/55 (9.1%)
    癸巳      : 5/55 (9.1%)
    癸丑      : 5/55 (9.1%)

[분석 2] 朝鮮國王 ↔ 堂子 동시 출현 비율

  전체 월수             : 758
  朝鮮國王 등장          : 55
  堂子 등장              : 59
  둘 다 등장             : 

In [7]:
# Cell 11: 논문 자료 정리 + 인포그래픽 (최종본)
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch

# ===== 경로 =====
CSV_DIR = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv"
FIG_DIR = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\figures"
os.makedirs(FIG_DIR, exist_ok=True)

INPUT_NER = os.path.join(CSV_DIR, "ner_results.csv")
INPUT_NONE_COOC = os.path.join(CSV_DIR, "joseon_none_cooccurrence.csv")
INPUT_RULES = os.path.join(CSV_DIR, "rules_month.csv")

OUT_MISSING = os.path.join(CSV_DIR, "joseon_missing_years.csv")
OUT_CRISIS_OVERLAP = os.path.join(CSV_DIR, "joseon_crisis_overlap.csv")
OUT_NONE_REVIEW = os.path.join(CSV_DIR, "joseon_none_for_review.csv")
OUT_INFOGRAPHIC = os.path.join(FIG_DIR, "10_infographic.png")

# ===== 폰트 =====
for fn in ['Malgun Gothic', 'SimSun', 'Microsoft YaHei', 'Batang']:
    if any(fn.lower() in f.name.lower() for f in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = fn
        print(f"[FONT] {fn}")
        break
plt.rcParams['axes.unicode_minus'] = False


# ============================================================
# [0] 강희 연간 주요 사건 (사용자 자료 기반)
# ============================================================
# 각 사건의 (시작연도, 종료연도, 명칭, 분류) — 종료=시작이면 단년 사건
EVENTS = [
    (1661, 1683, "천계령 및 해금령", "정책"),
    (1661, 1669, "오배 섭정 전횡", "내정"),
    (1661, 1683, "정씨 대만 정복전", "군사"),
    (1673, 1681, "삼번의 난", "군사"),
    (1675, 1675, "차하르 몽골 반란", "군사"),
    (1685, 1689, "야크사 전쟁·네르친스크 조약", "외교"),
    (1688, 1697, "준가르 전쟁 1기 (갈단)", "군사"),
    (1705, 1721, "중국 의례 논쟁", "외교"),
    (1708, 1712, "황태자 윤잉 폐립", "내정"),
    (1717, 1720, "준가르 티베트 침공·청 반격", "군사"),
]

def events_in_year(year):
    """해당 연도에 진행 중이던 사건 목록 반환"""
    return [name for (s, e, name, cat) in EVENTS if s <= year <= e]

def is_crisis_year(year):
    """위기 연도 여부 (사건 1개 이상 진행 중)"""
    return len(events_in_year(year)) > 0


# ============================================================
# [1] 朝鮮國王 부재 연도 추출
# ============================================================
print("\n" + "="*60)
print("[1] 朝鮮國王 부재 연도 분석")
print("="*60)

df = pd.read_csv(INPUT_NER)
joseon_df = df[df['word'] == '朝鮮國王']
joseon_years = sorted(joseon_df['period_year'].unique())
all_years = list(range(1661, 1723))
missing = sorted(set(all_years) - set(joseon_years))

# 부재 연도 + 진행 중 사건
missing_records = []
for y in missing:
    events = events_in_year(y)
    missing_records.append({
        'year': y,
        'reign_year': f"강희 {y - 1660}년",
        'is_crisis': len(events) > 0,
        'ongoing_events': "; ".join(events) if events else "(없음)"
    })
missing_df = pd.DataFrame(missing_records)
missing_df.to_csv(OUT_MISSING, index=False, encoding='utf-8-sig')

print(f"강희 연간 총 {len(all_years)}년 중")
print(f"  朝鮮國王 등장: {len(joseon_years)}년 ({len(joseon_years)/len(all_years)*100:.1f}%)")
print(f"  朝鮮國王 부재: {len(missing)}년 ({len(missing)/len(all_years)*100:.1f}%)")
print(f"\n[부재 연도 상세]")
print(missing_df.to_string(index=False))


# ============================================================
# [2] 위기 연도 vs 朝鮮國王 등장의 교차 분석
# ============================================================
print("\n" + "="*60)
print("[2] 위기 연도와 朝鮮國王 등장의 관계")
print("="*60)

crisis_records = []
for y in all_years:
    crisis_records.append({
        'year': y,
        'is_crisis': is_crisis_year(y),
        'joseon_present': y in joseon_years,
        'n_events': len(events_in_year(y)),
        'events': "; ".join(events_in_year(y))
    })
crisis_df = pd.DataFrame(crisis_records)
crisis_df.to_csv(OUT_CRISIS_OVERLAP, index=False, encoding='utf-8-sig')

# 2x2 교차표
ct = pd.crosstab(crisis_df['is_crisis'], crisis_df['joseon_present'],
                  margins=True, margins_name="합계")
ct.index = ['평시', '위기', '합계'][:len(ct)]
ct.columns = ['朝鮮國王 부재', '朝鮮國王 등장', '합계'][:len(ct.columns)]
print(f"\n[2x2 교차표]")
print(ct.to_string())

# 비율
n_crisis = crisis_df['is_crisis'].sum()
n_normal = (~crisis_df['is_crisis']).sum()
crisis_present = ((crisis_df['is_crisis']) & (crisis_df['joseon_present'])).sum()
normal_present = ((~crisis_df['is_crisis']) & (crisis_df['joseon_present'])).sum()

print(f"\n[등장률 비교]")
print(f"  위기 연도 ({n_crisis}년): 朝鮮國王 등장 {crisis_present}년 ({crisis_present/n_crisis*100:.1f}%)")
if n_normal > 0:
    print(f"  평시 연도 ({n_normal}년): 朝鮮國王 등장 {normal_present}년 ({normal_present/n_normal*100:.1f}%)")
else:
    print(f"  평시 연도 0년 — 강희 연간 전체가 어떤 사건과 겹침")

# 부재 연도 중 위기 비율
missing_crisis = sum(1 for y in missing if is_crisis_year(y))
print(f"\n[부재 연도의 성격]")
print(f"  부재 {len(missing)}년 중 위기 연도: {missing_crisis}년 ({missing_crisis/len(missing)*100:.1f}%)")
print(f"  부재 {len(missing)}년 중 평시 연도: {len(missing)-missing_crisis}년")


# ============================================================
# [3] NONE 단어 본인 분류용 자료
# ============================================================
print("\n" + "="*60)
print("[3] 朝鮮國王 동시출현 NONE 단어 검토 자료")
print("="*60)

none_df = pd.read_csv(INPUT_NONE_COOC)
top_none = none_df.sort_values(['Lift', 'n_TW'], ascending=[False, False]).head(30).copy()
top_none['my_category'] = ""
top_none['my_note'] = ""
top_none = top_none[['word', 'n_TW', 'n_W', 'P(W|T)', 'Jaccard', 'PMI', 'Lift',
                       'my_category', 'my_note']]
top_none.to_csv(OUT_NONE_REVIEW, index=False, encoding='utf-8-sig')

print(f"NONE 단어 Top 30 (lift 기준)")
pd.set_option('display.max_colwidth', 30)
pd.set_option('display.width', 200)
print(top_none[['word', 'n_TW', 'n_W', 'Lift', 'PMI']].to_string(index=False))
print(f"\n→ {OUT_NONE_REVIEW}")
print(f"  엑셀에서 my_category, my_note 컬럼을 직접 채우세요.")


# ============================================================
# [4] 인포그래픽
# ============================================================
print("\n" + "="*60)
print("[4] 인포그래픽 생성")
print("="*60)

rules_df = pd.read_csv(INPUT_RULES)
n_total_rules = len(rules_df)
mask = (rules_df['antecedents_tagged'].astype(str).str.contains('朝鮮國王', na=False) |
        rules_df['consequents_tagged'].astype(str).str.contains('朝鮮國王', na=False))
joseon_rules = rules_df[mask]

# 월별 분포 (NER에서 직접)
joseon_months = joseon_df['period_month'].unique()
month_dist = pd.Series(joseon_months).str.split('-').str[1].value_counts().sort_index()
all_months = [f"{i:02d}" for i in range(1, 13)]
counts = [int(month_dist.get(m, 0)) for m in all_months]

fig = plt.figure(figsize=(16, 12), facecolor='white')
gs = GridSpec(3, 3, figure=fig, hspace=0.6, wspace=0.35,
              left=0.06, right=0.96, top=0.92, bottom=0.07)

C_PRIMARY = '#8B0000'
C_SECONDARY = '#1F4E79'
C_CRISIS = '#D4AF37'
C_GRAY = '#666666'

fig.suptitle('朝鮮國王 in 康熙朝實錄 (1661-1722)\n'
             '연관 규칙 분석으로 본 정월 의례적 호명 패턴',
             fontsize=18, fontweight='bold', y=0.97, color='#222222')

# --- 패널 1: 핵심 통계 ---
ax1 = fig.add_subplot(gs[0, 0])
ax1.axis('off')
ax1.set_title('핵심 발견', fontsize=13, fontweight='bold', loc='left', pad=10)
stats_text = (
    f"분석 기간\n  康熙 元年-61年 (62년)\n\n"
    f"월별 트랜잭션\n  758개월\n\n"
    f"연관 규칙 총수\n  {n_total_rules:,}개\n\n"
    f"朝鮮國王 관련 규칙\n  {len(joseon_rules):,}개\n\n"
    f"최고 lift\n  13.44 (朝鮮國王↔堂子)\n\n"
    f"정월 집중도\n  100% ({counts[0]}/{sum(counts)})"
)
ax1.text(0.05, 0.95, stats_text, transform=ax1.transAxes,
         fontsize=10.5, verticalalignment='top', color='#333333', linespacing=1.6)

# --- 패널 2: 월별 분포 ---
ax2 = fig.add_subplot(gs[0, 1:])
colors = [C_PRIMARY if m == '01' else '#DDDDDD' for m in all_months]
ax2.bar(range(1, 13), counts, color=colors, edgecolor='white', linewidth=1.2)
ax2.set_xticks(range(1, 13))
ax2.set_xticklabels([f"{i}月" for i in range(1, 13)], fontsize=10)
ax2.set_ylabel('등장 월 수', fontsize=10)
ax2.set_title('朝鮮國王 등장 월의 분포 — 100% 정월 집중',
              fontsize=12, fontweight='bold', loc='left', pad=10)
ax2.text(1, max(counts) * 0.5, f"{counts[0]}회\n(100%)",
         ha='center', va='center', fontsize=14, fontweight='bold', color='white')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.grid(axis='y', alpha=0.3)

# --- 패널 3: 위기 사건 + 朝鮮國王 등장 타임라인 ---
ax3 = fig.add_subplot(gs[1, :])

# 배경: 위기 연도 음영
for (s, e, name, cat) in EVENTS:
    ax3.axvspan(s - 0.4, e + 0.4, ymin=0.45, ymax=0.95,
                alpha=0.12, color=C_CRISIS, zorder=1)

# 朝鮮國王 등장/부재 막대 (하단)
years = list(range(1661, 1723))
for y in years:
    color = C_PRIMARY if y in joseon_years else '#E0E0E0'
    ax3.bar(y, 0.35, bottom=0, color=color, width=0.85,
            edgecolor='none', zorder=2)

# 부재 연도 ✕ 마커
for y in missing:
    ax3.text(y, 0.18, '✕', ha='center', va='center',
             fontsize=11, color=C_SECONDARY, fontweight='bold', zorder=3)

# 사건 라벨 (상단, 사건 막대로)
event_y_positions = []
for i, (s, e, name, cat) in enumerate(EVENTS):
    y_pos = 0.55 + (i % 5) * 0.085  # 5단으로 분산
    ax3.plot([s, e], [y_pos, y_pos], color=C_CRISIS, linewidth=4,
             solid_capstyle='round', zorder=2)
    mid = (s + e) / 2
    ax3.text(mid, y_pos + 0.03, name, ha='center', va='bottom',
             fontsize=7.5, color='#5C4033', zorder=3)

ax3.set_xlim(1660, 1723)
ax3.set_ylim(0, 1.05)
ax3.set_yticks([])
ax3.set_xlabel('연도 (강희 1년 = 1661)', fontsize=10)
ax3.set_title(f'강희 연간 주요 사건과 朝鮮國王 등장 패턴 '
              f'(등장 {len(joseon_years)}년 · 부재 {len(missing)}년)',
              fontsize=12, fontweight='bold', loc='left', pad=10)
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)
ax3.spines['left'].set_visible(False)

legend_elements = [
    Patch(facecolor=C_PRIMARY, label='朝鮮國王 등장'),
    Patch(facecolor='#E0E0E0', label='朝鮮國王 부재 (✕)'),
    Patch(facecolor=C_CRISIS, alpha=0.4, label='위기 사건 진행 중'),
]
ax3.legend(handles=legend_elements, loc='lower right', fontsize=8,
           frameon=True, framealpha=0.9, ncol=3)

# --- 패널 4: 최강 규칙 Top 5 ---
ax4 = fig.add_subplot(gs[2, :2])
ax4.axis('off')
ax4.set_title('최강 연관 규칙 Top 5 (lift 기준)',
              fontsize=12, fontweight='bold', loc='left', pad=10)
top5 = joseon_rules.nlargest(5, 'lift')[['antecedents_tagged', 'consequents_tagged',
                                           'support', 'confidence', 'lift']]
table_text = ""
for i, row in enumerate(top5.itertuples(), 1):
    table_text += f"{i}. {row.antecedents_tagged}  →  {row.consequents_tagged}\n"
    table_text += f"     support={row.support:.4f}, confidence={row.confidence:.3f}, lift={row.lift:.2f}\n\n"
ax4.text(0.02, 0.95, table_text, transform=ax4.transAxes,
         fontsize=9.5, verticalalignment='top', family='monospace',
         color='#333333', linespacing=1.4)

# --- 패널 5: NONE Top 10 ---
ax5 = fig.add_subplot(gs[2, 2])
ax5.axis('off')
ax5.set_title('동시출현 NONE 단어 Top 10',
              fontsize=12, fontweight='bold', loc='left', pad=10)
top10 = top_none.head(10)
none_text = ""
for i, row in enumerate(top10.itertuples(), 1):
    none_text += f"{i:2d}. {row.word}  (lift {row.Lift:.1f})\n"
ax5.text(0.02, 0.95, none_text, transform=ax5.transAxes,
         fontsize=9.5, verticalalignment='top', family='monospace',
         color='#333333', linespacing=1.5)

fig.text(0.5, 0.015,
         "데이터: 국사편찬위원회 청실록 강희조 (1661-1722) | 분석: NER + FP-Growth | "
         "사건 분류: 사용자 종합",
         ha='center', fontsize=8, color=C_GRAY, style='italic')

plt.savefig(OUT_INFOGRAPHIC, dpi=200, bbox_inches='tight', facecolor='white')
plt.close()
print(f"[SAVED] {OUT_INFOGRAPHIC}")


# ============================================================
# 최종 요약
# ============================================================
print("\n" + "="*60)
print("[DONE] 논문 자료 정리 완료")
print("="*60)
print(f"\n생성 파일:")
print(f"  1. {OUT_MISSING}")
print(f"     → 부재 7년의 진행 중 사건 정리")
print(f"  2. {OUT_CRISIS_OVERLAP}")
print(f"     → 강희 62년 전체의 위기/평시 × 朝鮮國王 등장 매트릭스")
print(f"  3. {OUT_NONE_REVIEW}")
print(f"     → NONE 30개 본인 분류용 (my_category, my_note 채우기)")
print(f"  4. {OUT_INFOGRAPHIC}")
print(f"     → 인포그래픽 (논문/발표용)")


[FONT] Malgun Gothic

[1] 朝鮮國王 부재 연도 분석
강희 연간 총 62년 중
  朝鮮國王 등장: 55년 (88.7%)
  朝鮮國王 부재: 7년 (11.3%)

[부재 연도 상세]
 year reign_year  is_crisis                  ongoing_events
 1661      강희 1년       True  천계령 및 해금령; 오배 섭정 전횡; 정씨 대만 정복전
 1662      강희 2년       True  천계령 및 해금령; 오배 섭정 전횡; 정씨 대만 정복전
 1669      강희 9년       True  천계령 및 해금령; 오배 섭정 전횡; 정씨 대만 정복전
 1682     강희 22년       True            천계령 및 해금령; 정씨 대만 정복전
 1688     강희 28년       True 야크사 전쟁·네르친스크 조약; 준가르 전쟁 1기 (갈단)
 1696     강희 36년       True                  준가르 전쟁 1기 (갈단)
 1709     강희 49년       True             중국 의례 논쟁; 황태자 윤잉 폐립

[2] 위기 연도와 朝鮮國王 등장의 관계

[2x2 교차표]
    朝鮮國王 부재  朝鮮國王 등장  합계
평시        0        9   9
위기        7       46  53
합계        7       55  62

[등장률 비교]
  위기 연도 (53년): 朝鮮國王 등장 46년 (86.8%)
  평시 연도 (9년): 朝鮮國王 등장 9년 (100.0%)

[부재 연도의 성격]
  부재 7년 중 위기 연도: 7년 (100.0%)
  부재 7년 중 평시 연도: 0년

[3] 朝鮮國王 동시출현 NONE 단어 검토 자료
NONE 단어 Top 30 (lift 기준)
word  n_TW  n_W      Lift      PMI
  蒿齊    12   12 13.781818 3.784694
 九十三    1

C:\Users\USER\AppData\Local\Temp\ipykernel_2660\451177849.py:306: UserWarning: Glyph 10005 (\N{MULTIPLICATION X}) missing from font(s) Malgun Gothic.
  plt.savefig(OUT_INFOGRAPHIC, dpi=200, bbox_inches='tight', facecolor='white')
C:\Users\USER\AppData\Local\Temp\ipykernel_2660\451177849.py:306: UserWarning: Glyph 26397 (\N{CJK UNIFIED IDEOGRAPH-671D}) missing from font(s) DejaVu Sans Mono.
  plt.savefig(OUT_INFOGRAPHIC, dpi=200, bbox_inches='tight', facecolor='white')
C:\Users\USER\AppData\Local\Temp\ipykernel_2660\451177849.py:306: UserWarning: Glyph 39854 (\N{CJK UNIFIED IDEOGRAPH-9BAE}) missing from font(s) DejaVu Sans Mono.
  plt.savefig(OUT_INFOGRAPHIC, dpi=200, bbox_inches='tight', facecolor='white')
C:\Users\USER\AppData\Local\Temp\ipykernel_2660\451177849.py:306: UserWarning: Glyph 22283 (\N{CJK UNIFIED IDEOGRAPH-570B}) missing from font(s) DejaVu Sans Mono.
  plt.savefig(OUT_INFOGRAPHIC, dpi=200, bbox_inches='tight', facecolor='white')
C:\Users\USER\AppData\Local\Temp\ipykerne

[SAVED] C:\Users\USER\OneDrive\바탕 화면\nerproject\figures\10_infographic.png

[DONE] 논문 자료 정리 완료

생성 파일:
  1. C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\joseon_missing_years.csv
     → 부재 7년의 진행 중 사건 정리
  2. C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\joseon_crisis_overlap.csv
     → 강희 62년 전체의 위기/평시 × 朝鮮國王 등장 매트릭스
  3. C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\joseon_none_for_review.csv
     → NONE 30개 본인 분류용 (my_category, my_note 채우기)
  4. C:\Users\USER\OneDrive\바탕 화면\nerproject\figures\10_infographic.png
     → 인포그래픽 (논문/발표용)


In [8]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# 출력 잘림 방지
import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)


In [9]:
import pandas as pd
CSV_DIR = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv"

# 1) 부재 연도
print("="*60)
print("부재 7년")
print("="*60)
print(pd.read_csv(f"{CSV_DIR}\\joseon_missing_years.csv").to_string(index=False))

# 2) 위기 × 朝鮮國王 교차표
print("\n" + "="*60)
print("강희 62년 전체 매트릭스")
print("="*60)
df = pd.read_csv(f"{CSV_DIR}\\joseon_crisis_overlap.csv")
print(pd.crosstab(df['is_crisis'], df['joseon_present'],
                   margins=True, margins_name="합계").to_string())

# 3) 평시 연도
normal_years = df[~df['is_crisis']]['year'].tolist()
print(f"\n평시 연도 ({len(normal_years)}년): {normal_years}")
if normal_years:
    normal_with_joseon = df[(~df['is_crisis']) & (df['joseon_present'])]['year'].tolist()
    print(f"  └ 그중 朝鮮國王 등장: {normal_with_joseon}")

# 4) NONE Top 30 전체
print("\n" + "="*60)
print("NONE Top 30")
print("="*60)
none_review = pd.read_csv(f"{CSV_DIR}\\joseon_none_for_review.csv")
print(none_review[['word', 'n_TW', 'n_W', 'Lift', 'PMI']].to_string(index=False))


부재 7년
 year reign_year  is_crisis                  ongoing_events
 1661      강희 1년       True  천계령 및 해금령; 오배 섭정 전횡; 정씨 대만 정복전
 1662      강희 2년       True  천계령 및 해금령; 오배 섭정 전횡; 정씨 대만 정복전
 1669      강희 9년       True  천계령 및 해금령; 오배 섭정 전횡; 정씨 대만 정복전
 1682     강희 22년       True            천계령 및 해금령; 정씨 대만 정복전
 1688     강희 28년       True 야크사 전쟁·네르친스크 조약; 준가르 전쟁 1기 (갈단)
 1696     강희 36년       True                  준가르 전쟁 1기 (갈단)
 1709     강희 49년       True             중국 의례 논쟁; 황태자 윤잉 폐립

강희 62년 전체 매트릭스
joseon_present  False  True  합계
is_crisis                      
False               0     9   9
True                7    46  53
합계                  7    55  62

평시 연도 (9년): [1684, 1698, 1699, 1700, 1701, 1702, 1703, 1704, 1722]
  └ 그중 朝鮮國王 등장: [1684, 1698, 1699, 1700, 1701, 1702, 1703, 1704, 1722]

NONE Top 30
word  n_TW  n_W      Lift      PMI
  蒿齊    12   12 13.781818 3.784694
 九十三    10   10 13.781818 3.784694
四百八十     8    8 13.781818 3.784694
  徵課     8    8 13.781818 3.784694
三千八百     8 

In [16]:
# Cell 12: 조선 관련 단어 — 규칙 + 월 분포 (단순화)
import os
import pandas as pd

CSV_DIR = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv"
INPUT_RULES = os.path.join(CSV_DIR, "rules_month.csv")
INPUT_NER = os.path.join(CSV_DIR, "ner_results.csv")

OUT_RULES = os.path.join(CSV_DIR, "korea_rules.csv")
OUT_MONTH_DIST = os.path.join(CSV_DIR, "korea_month_distribution.csv")

# ============================================================
# [출력 1] 조선 관련 단어가 등장한 모든 규칙
# ============================================================
print("="*60)
print("[출력 1] 조선 관련 단어가 등장한 모든 규칙")
print("="*60)

rules = pd.read_csv(INPUT_RULES)

# 朝鮮이 antecedents 또는 consequents에 들어간 모든 규칙
mask = (rules['antecedents_tagged'].astype(str).str.contains('朝鮮', na=False) |
        rules['consequents_tagged'].astype(str).str.contains('朝鮮', na=False))
korea_rules = rules[mask].copy()
korea_rules = korea_rules.sort_values('lift', ascending=False).reset_index(drop=True)

print(f"\n전체 규칙: {len(rules):,}개")
print(f"조선 관련 규칙: {len(korea_rules):,}개")

korea_rules.to_csv(OUT_RULES, index=False, encoding='utf-8-sig')
print(f"[SAVED] {OUT_RULES}")

# 화면 출력 — 상위 30개
print(f"\n[조선 관련 규칙 Top 30 (lift 기준)]")
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.width', 200)
display_cols = ['antecedents_tagged', 'consequents_tagged', 'support', 'confidence', 'lift']
print(korea_rules[display_cols].head(30).to_string(index=False))


# ============================================================
# [출력 2] 조선 관련 단어가 등장한 월의 분포
# ============================================================
print("\n" + "="*60)
print("[출력 2] 조선 관련 단어가 등장한 월의 분포")
print("="*60)

ner = pd.read_csv(INPUT_NER)

# 조선 관련 단어 (朝鮮 포함)
korea_mask = ner['word'].astype(str).str.contains('朝鮮', na=False)
korea_ner = ner[korea_mask].copy()

# 단어 종류
korea_word_counts = korea_ner['word'].value_counts()
print(f"\n[조선 관련 단어 종류]")
print(korea_word_counts.to_string())

# 월별 분포 (윤달 제외 일반 1-12월)
korea_ner['month'] = korea_ner['period_month'].str.split('-').str[1]
korea_ner = korea_ner[~korea_ner['month'].str.contains('yun', na=False)]
korea_ner['month_int'] = korea_ner['month'].astype(int)

# 단어별 × 월별 교차표
month_pivot = pd.crosstab(korea_ner['word'], korea_ner['month_int'], margins=True, margins_name='합계')
print(f"\n[단어별 × 월별 등장 횟수]")
print(month_pivot.to_string())

# 전체 합산 월 분포
total_by_month = korea_ner['month_int'].value_counts().sort_index()
print(f"\n[조선 관련 단어 전체 월별 분포]")
for m in range(1, 13):
    cnt = int(total_by_month.get(m, 0))
    bar = '█' * cnt
    print(f"  {m:2d}월: {cnt:4d}회  {bar}")

# CSV 저장
month_pivot.to_csv(OUT_MONTH_DIST, encoding='utf-8-sig')
print(f"\n[SAVED] {OUT_MONTH_DIST}")

print("\n" + "="*60)
print("[DONE]")
print("="*60)


[출력 1] 조선 관련 단어가 등장한 모든 규칙

전체 규칙: 1,475개
조선 관련 규칙: 273개
[SAVED] C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\korea_rules.csv

[조선 관련 규칙 Top 30 (lift 기준)]
  antecedents_tagged   consequents_tagged  support  confidence      lift
           朝鮮國王(PER)     堂子(LOC), 滿洲(GRP) 0.051451    0.709091 13.437273
    堂子(LOC), 滿洲(GRP)            朝鮮國王(PER) 0.051451    0.975000 13.437273
    八旗(SYS), 堂子(LOC)            朝鮮國王(PER) 0.052770    0.952381 13.125541
           朝鮮國王(PER)     八旗(SYS), 堂子(LOC) 0.052770    0.727273 13.125541
           朝鮮國王(PER)    堂子(LOC), 正月(DATE) 0.071240    0.981818 13.056459
           朝鮮國王(PER)    元旦(DATE), 堂子(LOC) 0.071240    0.981818 13.056459
           朝鮮國王(PER)     堂子(LOC), 朔(DATE) 0.071240    0.981818 13.056459
   堂子(LOC), 正月(DATE)            朝鮮國王(PER) 0.071240    0.947368 13.056459
   元旦(DATE), 堂子(LOC)            朝鮮國王(PER) 0.071240    0.947368 13.056459
    堂子(LOC), 朔(DATE)            朝鮮國王(PER) 0.071240    0.947368 13.056459
    堂子(LOC), 藩王(OFF)            朝鮮國王(PER) 0

In [19]:
# '朝鮮'만 나온 월 알아보기.

import pandas as pd
df = pd.read_csv(r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\ner_results.csv")
print(df[df['word']=='朝鮮'][['period_year','period_month','freq']])

import pandas as pd

CSV_DIR = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv"

# 분석용 (NONE/UNK 제외)
df = pd.read_csv(rf"{CSV_DIR}\ner_results.csv")
# 전체 (NONE 포함)
df_all = pd.read_csv(rf"{CSV_DIR}\ner_results_all.csv")

target = "1711-03"  # 강희 11년 3월

print(f"="*60)
print(f"[{target}] 분석용 데이터 (NONE/UNK 제외)")
print(f"="*60)
sub = df[df['period_month'] == target].sort_values('freq', ascending=False)
print(f"단어 수: {len(sub)}")
print(sub[['word','category','freq']].to_string(index=False))

print(f"\n{'='*60}")
print(f"[{target}] 전체 데이터 (NONE 포함)")
print(f"{'='*60}")
sub_all = df_all[df_all['period_month'] == target].sort_values('freq', ascending=False)
print(f"단어 수: {len(sub_all)}")
print(sub_all[['word','category','freq']].to_string(index=False))


       period_year period_month  freq
17238         1711      1711-03     1
[1711-03] 분석용 데이터 (NONE/UNK 제외)
단어 수: 54
word category  freq
  侍讀      OFF     3
  內閣      OFF     2
  學士      OFF     2
  巡撫      OFF     2
   爵      OFF     2
  徐元      PER     2
  兗州      LOC     1
  交趾      LOC     1
  九卿      OFF     1
公大學士      OFF     1
吳巴達子      PER     1
  哈密      LOC     1
  兵部      OFF     1
  南巡      EVT     1
   堯      PER     1
  國公      OFF     1
 大學士      OFF     1
  壬辰     DATE     1
  山東      LOC     1
  己丑     DATE     1
  三逆      EVT     1
  三月     DATE     1
  庚寅     DATE     1
  帝堯      PER     1
 張鵬翮      PER     1
 張伯行      PER     1
 明洪武     DATE     1
 暢春園      LOC     1
  朝鮮      LOC     1
  提鎮      OFF     1
  武陵      LOC     1
  江寧      LOC     1
  浙江      LOC     1
  浙省      LOC     1
   淮      LOC     1
  清明     DATE     1
  湖南      LOC     1
  桑乾      LOC     1
  滹沱      LOC     1
   漕      SYS     1
   漳      LOC     1
  漢人      GRP     1
 烏思藏      LOC     1
  特

In [20]:
# Cell 13: 다른 외번국 (蒙古 제외) — 규칙 + 월 분포
import os
import pandas as pd

CSV_DIR = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv"
INPUT_RULES = os.path.join(CSV_DIR, "rules_month.csv")
INPUT_NER_ALL = os.path.join(CSV_DIR, "ner_results_all.csv")
INPUT_NER = os.path.join(CSV_DIR, "ner_results.csv")

OUT_RULES = os.path.join(CSV_DIR, "other_vassal_rules.csv")
OUT_MONTH_DIST = os.path.join(CSV_DIR, "other_vassal_month_distribution.csv")

# ============================================================
# 외번국 단어 그룹 정의 (蒙古 제외)
# ============================================================
df_all = pd.read_csv(INPUT_NER_ALL)

vassal_patterns = {
    '琉球':   r'琉球',
    '安南':   r'安南',
    '哈薩克': r'哈薩克',
    '準噶爾': r'準噶爾|準噶尔|準噶|噶爾丹',
    '達賴':   r'達賴|喇嘛',
}

vassal_word_map = {}  # word -> vassal_name
for name, pat in vassal_patterns.items():
    words = df_all[df_all['word'].astype(str).str.contains(pat, na=False, regex=True)]['word'].unique()
    for w in words:
        vassal_word_map[w] = name

print(f"외번국 그룹: {list(vassal_patterns.keys())}")
for name in vassal_patterns:
    cnt = sum(1 for v in vassal_word_map.values() if v == name)
    print(f"  {name}: {cnt}개 단어")
all_vassal_words = set(vassal_word_map.keys())
print(f"전체 외번 단어 수: {len(all_vassal_words)}")


# ============================================================
# [출력 1] 외번국 단어가 등장한 모든 규칙
# ============================================================
print("\n" + "="*60)
print("[출력 1] 외번국 단어가 등장한 모든 규칙")
print("="*60)

rules = pd.read_csv(INPUT_RULES)

# 규칙의 antecedents/consequents에서 외번 단어 포함 여부 확인
def find_vassals_in_text(text):
    """규칙 문자열에서 어떤 외번이 포함됐는지 반환"""
    if pd.isna(text):
        return []
    found = []
    for w, name in vassal_word_map.items():
        if w in str(text):
            found.append(name)
    return list(set(found))

rules['vassals_found'] = rules.apply(
    lambda r: list(set(find_vassals_in_text(r['antecedents_tagged']) + 
                        find_vassals_in_text(r['consequents_tagged']))),
    axis=1
)
rules['has_vassal'] = rules['vassals_found'].apply(len) > 0

vassal_rules = rules[rules['has_vassal']].copy()
vassal_rules['vassal_label'] = vassal_rules['vassals_found'].apply(lambda x: ', '.join(sorted(x)))
vassal_rules = vassal_rules.sort_values('lift', ascending=False).reset_index(drop=True)

print(f"\n전체 규칙: {len(rules):,}개")
print(f"외번국(蒙古 제외) 포함 규칙: {len(vassal_rules):,}개")

# 외번별 규칙 수
print(f"\n[외번별 규칙 수]")
vassal_counts = {}
for vlist in vassal_rules['vassals_found']:
    for v in vlist:
        vassal_counts[v] = vassal_counts.get(v, 0) + 1
for v in sorted(vassal_counts.keys()):
    print(f"  {v}: {vassal_counts[v]}개")

# 저장
out_cols = ['vassal_label', 'antecedents_tagged', 'consequents_tagged',
            'support', 'confidence', 'lift']
vassal_rules[out_cols].to_csv(OUT_RULES, index=False, encoding='utf-8-sig')
print(f"\n[SAVED] {OUT_RULES}")

# 화면 출력 — 외번별로 묶어서
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.width', 200)

for vname in sorted(vassal_counts.keys()):
    sub = vassal_rules[vassal_rules['vassals_found'].apply(lambda x: vname in x)]
    print(f"\n{'─'*60}")
    print(f"[{vname}] 규칙 {len(sub)}개 (Top 15 by lift)")
    print('─'*60)
    print(sub[['antecedents_tagged','consequents_tagged','support','confidence','lift']]
            .head(15).to_string(index=False))


# ============================================================
# [출력 2] 외번국 단어의 월별 등장 분포
# ============================================================
print("\n" + "="*60)
print("[출력 2] 외번국 단어의 월별 등장 분포")
print("="*60)

ner = pd.read_csv(INPUT_NER)  # 분석용 (NONE/UNK 제외)

# 외번 단어만 추출
ner_vassal = ner[ner['word'].isin(all_vassal_words)].copy()
ner_vassal['vassal'] = ner_vassal['word'].map(vassal_word_map)
ner_vassal['month'] = ner_vassal['period_month'].str.split('-').str[1]
ner_vassal = ner_vassal[~ner_vassal['month'].str.contains('yun', na=False)]
ner_vassal['month_int'] = ner_vassal['month'].astype(int)

# 외번별 × 월별 (등장 월 수 기준 — 같은 달에 여러 단어 있어도 1로 카운트)
month_counts = ner_vassal.groupby(['vassal', 'period_month']).size().reset_index(name='n')
month_counts['month_int'] = month_counts['period_month'].str.split('-').str[1].astype(int)

pivot = pd.crosstab(month_counts['vassal'], month_counts['month_int'], margins=True, margins_name='합계')
print(f"\n[외번별 × 월별 등장 월 수]")
print(pivot.to_string())

# 외번별 막대 그래프 (텍스트)
print(f"\n[외번별 월 분포 시각화]")
for vname in sorted(vassal_patterns.keys()):
    sub = month_counts[month_counts['vassal'] == vname]
    if len(sub) == 0:
        continue
    print(f"\n  ── {vname} (총 {len(sub)}개월) ──")
    by_month = sub.groupby('month_int').size()
    for m in range(1, 13):
        cnt = int(by_month.get(m, 0))
        bar = '█' * cnt
        print(f"    {m:2d}월: {cnt:3d}회  {bar}")

# 전체 합산
print(f"\n[외번 전체 합산 월 분포 — 어느 달에 외번이 자주 나오는가]")
total = month_counts.groupby('month_int').size()
for m in range(1, 13):
    cnt = int(total.get(m, 0))
    bar = '█' * cnt
    print(f"  {m:2d}월: {cnt:3d}회  {bar}")

# 저장
pivot.to_csv(OUT_MONTH_DIST, encoding='utf-8-sig')
print(f"\n[SAVED] {OUT_MONTH_DIST}")

print("\n" + "="*60)
print("[DONE]")
print("="*60)


외번국 그룹: ['琉球', '安南', '哈薩克', '準噶爾', '達賴']
  琉球: 1개 단어
  安南: 1개 단어
  哈薩克: 1개 단어
  準噶爾: 107개 단어
  達賴: 40개 단어
전체 외번 단어 수: 150

[출력 1] 외번국 단어가 등장한 모든 규칙

전체 규칙: 1,475개
외번국(蒙古 제외) 포함 규칙: 0개

[외번별 규칙 수]

[SAVED] C:\Users\USER\OneDrive\바탕 화면\nerproject\csv\other_vassal_rules.csv

[출력 2] 외번국 단어의 월별 등장 분포

[외번별 × 월별 등장 월 수]
month_int  1  2  3  4  5  6   7  8  9  10  11  12  합계
vassal                                               
哈薩克        0  0  0  0  0  0   0  0  0   1   0   0   1
安南         0  0  1  1  1  1   0  0  1   0   1   1   7
準噶爾        1  0  1  2  3  4   4  6  2   1   1   1  26
琉球         0  0  1  0  0  0   1  0  1   3   0   0   6
達賴         0  1  0  2  0  4   5  3  0   2   1   0  18
합계         1  1  3  5  4  9  10  9  4   7   3   2  58

[외번별 월 분포 시각화]

  ── 哈薩克 (총 1개월) ──
     1월:   0회  
     2월:   0회  
     3월:   0회  
     4월:   0회  
     5월:   0회  
     6월:   0회  
     7월:   0회  
     8월:   0회  
     9월:   0회  
    10월:   1회  █
    11월:   0회  
    12월:   0회  

  ── 安南 (총 7개월) ──
  

In [21]:
# Cell 14: 蒙古 — 의례(정월) vs 비의례(비정월) 분리 분석
import os
import pandas as pd

CSV_DIR = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv"
INPUT_RULES = os.path.join(CSV_DIR, "rules_month.csv")
INPUT_NER = os.path.join(CSV_DIR, "ner_results.csv")
INPUT_NER_ALL = os.path.join(CSV_DIR, "ner_results_all.csv")

OUT_RULES_RITUAL = os.path.join(CSV_DIR, "mongol_rules_ritual.csv")
OUT_RULES_NONRITUAL = os.path.join(CSV_DIR, "mongol_rules_nonritual.csv")
OUT_MONTH_DIST = os.path.join(CSV_DIR, "mongol_month_distribution.csv")

# ============================================================
# 蒙古 단어 그룹 정의 (몽골 부족명 포함)
# ============================================================
MONGOL_DIRECT = ['蒙古', '內蒙古', '外蒙古']
MONGOL_TRIBES = ['科爾沁','喀爾喀','土默特','翁牛特','巴林','敖漢','扎魯特',
                  '杜爾伯特','喀喇沁','毛明安','哈納','察哈爾','鄂爾多斯',
                  '蘇尼特','烏珠穆沁','奈曼','阿巴噶','茂明安','克什克騰']

df_all = pd.read_csv(INPUT_NER_ALL)

# 실제 데이터에 있는 단어만 필터
mongol_words = set()
print("[蒙古 그룹 단어 확인]")
for w in MONGOL_DIRECT + MONGOL_TRIBES:
    if w in df_all['word'].values:
        cat = df_all[df_all['word']==w]['category'].iloc[0]
        n = (df_all['word']==w).sum()
        mongol_words.add(w)
        print(f"  {w}: {n}회, NER={cat}")
    # 부분 매칭도 추가 (蒙古로 시작하는 단어들)
mongol_partial = df_all[df_all['word'].astype(str).str.contains('蒙古', na=False)]['word'].unique()
for w in mongol_partial:
    mongol_words.add(w)

print(f"\n蒙古 그룹 총 단어 수: {len(mongol_words)}")


# ============================================================
# [출력 1] 蒙古 등장 규칙을 의례/비의례로 분리
# ============================================================
print("\n" + "="*60)
print("[출력 1-1] 蒙古 포함 규칙 추출")
print("="*60)

rules = pd.read_csv(INPUT_RULES)

def has_mongol_in_text(text):
    if pd.isna(text):
        return False
    text = str(text)
    return any(w in text for w in mongol_words)

rules['has_mongol'] = rules.apply(
    lambda r: has_mongol_in_text(r['antecedents_tagged']) or 
                has_mongol_in_text(r['consequents_tagged']),
    axis=1
)
mongol_rules = rules[rules['has_mongol']].copy()
print(f"전체 규칙: {len(rules):,}개")
print(f"蒙古 그룹 포함 규칙: {len(mongol_rules):,}개")

# 의례 정형구가 있는지로 의례/비의례 분리
RITUAL_MARKERS = ['正月','元旦','朔','堂子','萬壽節','拜神','貢禮物','進歲',
                   '還宮','表賀','禮御殿','藩王','朝鮮國王']

def is_ritual_rule(row):
    text = str(row['antecedents_tagged']) + ' ' + str(row['consequents_tagged'])
    return any(m in text for m in RITUAL_MARKERS)

mongol_rules['is_ritual'] = mongol_rules.apply(is_ritual_rule, axis=1)

ritual_rules = mongol_rules[mongol_rules['is_ritual']].sort_values('lift', ascending=False)
nonritual_rules = mongol_rules[~mongol_rules['is_ritual']].sort_values('lift', ascending=False)

print(f"\n  의례 규칙 (정월 의례 정형구 동반): {len(ritual_rules)}개")
print(f"  비의례 규칙 (의례 정형구 없음):    {len(nonritual_rules)}개")

# 저장
out_cols = ['antecedents_tagged','consequents_tagged','support','confidence','lift']
ritual_rules[out_cols].to_csv(OUT_RULES_RITUAL, index=False, encoding='utf-8-sig')
nonritual_rules[out_cols].to_csv(OUT_RULES_NONRITUAL, index=False, encoding='utf-8-sig')

# 화면 출력
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.width', 200)

print("\n" + "─"*60)
print(f"[의례 규칙 Top 20] (lift 기준)")
print("─"*60)
print(ritual_rules[out_cols].head(20).to_string(index=False))

print("\n" + "─"*60)
print(f"[비의례 규칙 Top 20] (lift 기준)")
print("─"*60)
print(nonritual_rules[out_cols].head(20).to_string(index=False))

print(f"\n[SAVED] {OUT_RULES_RITUAL}")
print(f"[SAVED] {OUT_RULES_NONRITUAL}")


# ============================================================
# [출력 2] 蒙古 단어의 월별 등장 분포 (의례/비의례 비교)
# ============================================================
print("\n" + "="*60)
print("[출력 2] 蒙古 그룹의 월별 등장 분포")
print("="*60)

ner = pd.read_csv(INPUT_NER)
ner_mongol = ner[ner['word'].isin(mongol_words)].copy()
ner_mongol['month'] = ner_mongol['period_month'].str.split('-').str[1]
ner_mongol = ner_mongol[~ner_mongol['month'].str.contains('yun', na=False)]
ner_mongol['month_int'] = ner_mongol['month'].astype(int)

# 등장 월 (중복 제거)
unique_months = ner_mongol[['period_month','month_int']].drop_duplicates()
total_months = len(unique_months)
print(f"\n蒙古 그룹 등장 월 수: {total_months}개월")

# 정월 vs 비정월 분리
jan_months = unique_months[unique_months['month_int']==1]
nonjan_months = unique_months[unique_months['month_int']!=1]
print(f"  정월(의례): {len(jan_months)}개월 ({len(jan_months)/total_months*100:.1f}%)")
print(f"  비정월(비의례): {len(nonjan_months)}개월 ({len(nonjan_months)/total_months*100:.1f}%)")

# 월별 분포
by_month = unique_months['month_int'].value_counts().sort_index()
print(f"\n[월별 등장 월 수]")
for m in range(1, 13):
    cnt = int(by_month.get(m, 0))
    bar = '█' * cnt
    label = " ← 정월(의례)" if m == 1 else ""
    print(f"  {m:2d}월: {cnt:3d}회  {bar}{label}")

# 단어별 × 월별 (어떤 부족이 어느 달에 나오는가)
print(f"\n[단어별 × 정월/비정월 등장 월 수]")
ner_mongol['period'] = ner_mongol['month_int'].apply(lambda m: '정월' if m==1 else '비정월')
word_period = (ner_mongol.groupby(['word','period'])['period_month']
                .nunique().unstack(fill_value=0))
if '정월' in word_period.columns and '비정월' in word_period.columns:
    word_period['합계'] = word_period['정월'] + word_period['비정월']
    word_period['정월비율'] = (word_period['정월']/word_period['합계']*100).round(1)
    word_period = word_period.sort_values('합계', ascending=False)
    print(word_period.to_string())

# 저장
word_period.to_csv(OUT_MONTH_DIST, encoding='utf-8-sig')
print(f"\n[SAVED] {OUT_MONTH_DIST}")


# ============================================================
# [요약] 朝鮮 vs 蒙古 비교
# ============================================================
print("\n" + "="*60)
print("[참고] 朝鮮國王과의 비교")
print("="*60)
print(f"""
                朝鮮國王         蒙古 그룹
─────────────────────────────────────────────
등장 월 수      55개월           {total_months}개월
정월 집중도     100% (55/55)    {len(jan_months)/total_months*100:.1f}% ({len(jan_months)}/{total_months})
비정월 등장     0회              {len(nonjan_months)}회
패턴            단일 (의례)      {'이중 (의례+비의례)' if len(nonjan_months)>0 else '단일 (의례)'}

→ 의례 규칙: {len(ritual_rules)}개
→ 비의례 규칙: {len(nonritual_rules)}개
""")

print("="*60)
print("[DONE]")
print("="*60)


[蒙古 그룹 단어 확인]
  蒙古: 153회, NER=GRP
  科爾沁: 63회, NER=GRP
  土默特: 33회, NER=GRP
  翁牛特: 39회, NER=GRP
  巴林: 43회, NER=GRP
  敖漢: 11회, NER=GRP
  扎魯特: 11회, NER=GRP
  杜爾伯特: 11회, NER=GRP
  喀喇沁: 19회, NER=GRP
  毛明安: 10회, NER=PER
  哈納: 17회, NER=PER
  察哈爾: 3회, NER=GRP
  奈曼: 3회, NER=GRP

蒙古 그룹 총 단어 수: 14

[출력 1-1] 蒙古 포함 규칙 추출
전체 규칙: 1,475개
蒙古 그룹 포함 규칙: 88개

  의례 규칙 (정월 의례 정형구 동반): 36개
  비의례 규칙 (의례 정형구 없음):    52개

────────────────────────────────────────────────────────────
[의례 규칙 Top 20] (lift 기준)
────────────────────────────────────────────────────────────
antecedents_tagged consequents_tagged  support  confidence      lift
 正月(DATE), 蒙古(GRP)           元旦(DATE) 0.050132    0.926829 11.151374
          元旦(DATE)  正月(DATE), 蒙古(GRP) 0.050132    0.603175 11.151374
          正月(DATE)  元旦(DATE), 蒙古(GRP) 0.050132    0.558824 11.147059
 元旦(DATE), 蒙古(GRP)           正月(DATE) 0.050132    1.000000 11.147059
 元旦(DATE), 蒙古(GRP)            八旗(SYS) 0.050132    1.000000  9.243902
  八旗(SYS), 蒙古(GRP)           元旦(DATE) 0.

In [22]:
# Cell 15: 외번 비교 시각화 — 시안
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch

CSV_DIR = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\csv"
FIG_DIR = r"C:\Users\USER\OneDrive\바탕 화면\nerproject\figures"
os.makedirs(FIG_DIR, exist_ok=True)

OUT_FIG = os.path.join(FIG_DIR, "11_vassal_comparison.png")

# 폰트
for fn in ['Malgun Gothic', 'SimSun', 'Microsoft YaHei', 'Batang']:
    if any(fn.lower() in f.name.lower() for f in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = fn
        print(f"[FONT] {fn}")
        break
plt.rcParams['axes.unicode_minus'] = False


# ============================================================
# 데이터 (이전 분석 결과 직접 입력 — 재계산 비용 회피)
# ============================================================
# (외번명, 등장 월 수, 정월 등장 월 수, 카테고리)
data = [
    # 朝鮮
    ('朝鮮國王',     55, 55, 'korea'),
    
    # 蒙古 — 그룹 단위 + 부족별
    ('蒙古',        150, 38, 'mongol_general'),
    ('科爾沁',       61, 25, 'mongol_tribe'),
    ('巴林',         43, 26, 'mongol_tribe'),
    ('翁牛特',       38, 23, 'mongol_tribe'),
    ('土默特',       32, 21, 'mongol_tribe'),
    ('喀喇沁',       19, 10, 'mongol_tribe'),
    ('哈納',         17, 10, 'mongol_tribe'),
    ('杜爾伯特',     11, 10, 'mongol_tribe'),
    ('敖漢',         11, 10, 'mongol_tribe'),
    ('扎魯特',       11, 10, 'mongol_tribe'),
    ('毛明安',       10, 10, 'mongol_tribe'),
    
    # 다른 외번
    ('準噶爾',       26,  1, 'other'),
    ('達賴',         18,  0, 'other'),
    ('安南',          7,  0, 'other'),
    ('琉球',          6,  0, 'other'),
    ('哈薩克',        1,  0, 'other'),
]

df = pd.DataFrame(data, columns=['name', 'total', 'jan', 'cat'])
df['nonjan'] = df['total'] - df['jan']
df['jan_ratio'] = df['jan'] / df['total'] * 100

# 색상
COLOR = {
    'korea':          '#8B0000',  # 진한 적색
    'mongol_general': '#1F4E79',  # 진한 청색
    'mongol_tribe':   '#5B9BD5',  # 연한 청색
    'other':          '#7F7F7F',  # 회색
}
LABEL = {
    'korea': '朝鮮',
    'mongol_general': '蒙古 (총칭)',
    'mongol_tribe': '蒙古 부족',
    'other': '기타 외번',
}


# ============================================================
# 그림 구성
# ============================================================
fig = plt.figure(figsize=(16, 11), facecolor='white')
gs = GridSpec(2, 2, figure=fig, hspace=0.35, wspace=0.3,
              left=0.07, right=0.97, top=0.92, bottom=0.08)

fig.suptitle('강희조 외번의 등장 패턴 비교 — 朝鮮의 의례적 단일성 vs 蒙古의 이중성',
             fontsize=16, fontweight='bold', y=0.97)


# --- 패널 A: 산점도 (정월 비율 × 등장 월 수) ---
ax_a = fig.add_subplot(gs[0, :])

for cat in ['other', 'mongol_tribe', 'mongol_general', 'korea']:
    sub = df[df['cat'] == cat]
    ax_a.scatter(sub['jan_ratio'], sub['total'],
                  s=sub['total'] * 8 + 50,
                  c=COLOR[cat], alpha=0.75,
                  edgecolors='white', linewidth=1.5,
                  label=LABEL[cat], zorder=3)

# 라벨
for _, row in df.iterrows():
    offset_x, offset_y = 2, 2
    if row['name'] == '朝鮮國王':
        offset_x, offset_y = -3, 5
        ha = 'right'
    elif row['name'] == '蒙古':
        offset_x, offset_y = 3, 5
        ha = 'left'
    else:
        ha = 'left'
    ax_a.annotate(row['name'],
                   (row['jan_ratio'], row['total']),
                   xytext=(offset_x, offset_y),
                   textcoords='offset points',
                   fontsize=10, ha=ha,
                   color='#222', zorder=4)

# 영역 구획선
ax_a.axvline(x=50, color='#CCCCCC', linestyle='--', linewidth=1, zorder=1)
ax_a.axhline(y=30, color='#CCCCCC', linestyle='--', linewidth=1, zorder=1)

# 영역 라벨
ax_a.text(95, 60, '의례 전용·고빈도\n(朝鮮)', ha='right', va='center',
           fontsize=9, color='#8B0000', alpha=0.7, fontweight='bold')
ax_a.text(95, 12, '의례 전용·저빈도\n(소규모 몽골 부족)', ha='right', va='center',
           fontsize=9, color='#5B9BD5', alpha=0.7)
ax_a.text(5, 60, '일상 통치 중심·고빈도\n(蒙古, 科爾沁)', ha='left', va='center',
           fontsize=9, color='#1F4E79', alpha=0.7)
ax_a.text(5, 12, '사건 기반·저빈도\n(準噶爾·達賴 등)', ha='left', va='center',
           fontsize=9, color='#7F7F7F', alpha=0.7)

ax_a.set_xlabel('정월 집중도 (%)', fontsize=11)
ax_a.set_ylabel('등장 월 수', fontsize=11)
ax_a.set_xlim(-5, 105)
ax_a.set_ylim(-5, 75)
ax_a.set_title('A. 외번별 등장 패턴 — 정월 집중도와 등장 빈도의 4분면',
                fontsize=12, fontweight='bold', loc='left', pad=10)
ax_a.legend(loc='upper center', fontsize=9, frameon=True, ncol=4,
             bbox_to_anchor=(0.5, -0.13))
ax_a.grid(True, alpha=0.3)
ax_a.spines['top'].set_visible(False)
ax_a.spines['right'].set_visible(False)


# --- 패널 B: 朝鮮 vs 蒙古 부족별 stacked bar ---
ax_b = fig.add_subplot(gs[1, 0])

# 朝鮮 + 蒙古 그룹만 정월 비율 순으로
mongol_korea = df[df['cat'].isin(['korea', 'mongol_general', 'mongol_tribe'])].copy()
mongol_korea = mongol_korea.sort_values('jan_ratio', ascending=True)

y_pos = np.arange(len(mongol_korea))
bar_colors = [COLOR[c] for c in mongol_korea['cat']]

bars1 = ax_b.barh(y_pos, mongol_korea['jan'], color=bar_colors,
                    edgecolor='white', linewidth=1, label='정월(의례)')
bars2 = ax_b.barh(y_pos, mongol_korea['nonjan'], left=mongol_korea['jan'],
                    color='#E0E0E0', edgecolor='white', linewidth=1, label='비정월(일상)')

# 정월 비율 라벨
for i, row in enumerate(mongol_korea.itertuples()):
    ax_b.text(row.total + 1.5, i, f"{row.jan_ratio:.0f}%",
               va='center', fontsize=9, color='#444')

ax_b.set_yticks(y_pos)
ax_b.set_yticklabels(mongol_korea['name'], fontsize=10)
ax_b.set_xlabel('등장 월 수', fontsize=11)
ax_b.set_title('B. 朝鮮 vs 蒙古 — 정월 집중도 위계',
                fontsize=12, fontweight='bold', loc='left', pad=10)
ax_b.legend(loc='lower right', fontsize=9, frameon=True)
ax_b.spines['top'].set_visible(False)
ax_b.spines['right'].set_visible(False)
ax_b.grid(axis='x', alpha=0.3)
ax_b.set_xlim(0, max(mongol_korea['total']) * 1.15)


# --- 패널 C: 외번별 월 분포 라인 ---
ax_c = fig.add_subplot(gs[1, 1])

# 사전 계산된 월별 분포 (이전 분석 결과)
month_data = {
    '朝鮮國王':  [55, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    '蒙古 그룹':  [42, 13, 15, 16, 13, 19, 21, 24, 12, 17, 12, 15],
    '準噶爾':    [1, 0, 1, 2, 3, 4, 4, 6, 2, 1, 1, 1],
    '達賴':      [0, 1, 0, 2, 0, 4, 5, 3, 0, 2, 1, 0],
}

months = list(range(1, 13))
line_colors = {
    '朝鮮國王': '#8B0000',
    '蒙古 그룹': '#1F4E79',
    '準噶爾':   '#7F7F7F',
    '達賴':    '#A0A0A0',
}
line_styles = {
    '朝鮮國王': '-',
    '蒙古 그룹': '-',
    '準噶爾':   '--',
    '達賴':    ':',
}

for name, vals in month_data.items():
    ax_c.plot(months, vals, marker='o', markersize=6,
                linewidth=2.2, label=name,
                color=line_colors[name], linestyle=line_styles[name])

# 朝鮮國왕 정점 강조
ax_c.annotate(f'정월 55회\n(100%)',
                xy=(1, 55), xytext=(2.5, 50),
                fontsize=10, fontweight='bold', color='#8B0000',
                arrowprops=dict(arrowstyle='->', color='#8B0000', lw=1.2))

ax_c.set_xticks(months)
ax_c.set_xticklabels([f"{m}월" for m in months], fontsize=9)
ax_c.set_xlabel('월', fontsize=11)
ax_c.set_ylabel('등장 월 수', fontsize=11)
ax_c.set_title('C. 외번별 월 분포 — 朝鮮은 정월 단일 봉우리',
                fontsize=12, fontweight='bold', loc='left', pad=10)
ax_c.legend(loc='upper right', fontsize=9, frameon=True)
ax_c.spines['top'].set_visible(False)
ax_c.spines['right'].set_visible(False)
ax_c.grid(True, alpha=0.3)


# 푸터
fig.text(0.5, 0.015,
         "데이터: 국사편찬위원회 청실록 강희조 (1661-1722, 762개월) | 분석: NER + FP-Growth",
         ha='center', fontsize=8, color='#666', style='italic')

plt.savefig(OUT_FIG, dpi=200, bbox_inches='tight', facecolor='white')
plt.close()
print(f"[SAVED] {OUT_FIG}")


[FONT] Malgun Gothic


Text(0.5, 0.97, '강희조 외번의 등장 패턴 비교 — 朝鮮의 의례적 단일성 vs 蒙古의 이중성')

Text(-3, 5, '朝鮮國王')

Text(3, 5, '蒙古')

Text(2, 2, '科爾沁')

...

Text(0.5, 0, '월')

Text(0, 0.5, '등장 월 수')

Text(0.0, 1.0, 'C. 외번별 월 분포 — 朝鮮은 정월 단일 봉우리')

Text(0.5, 0.015, '데이터: 국사편찬위원회 청실록 강희조 (1661-1722, 762개월) | 분석: NER + FP-Growth')

[SAVED] C:\Users\USER\OneDrive\바탕 화면\nerproject\figures\11_vassal_comparison.png
